# IV-6 独立成分分析から ILRMA へ（ノートブック版）

教科書の IV-6 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約 5 分（5 節を試しの設定にしたとき）。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 実数の ICA：中心極限定理・白色化・FastICA | B | `ica_real.py` |
| 2 | 補助関数と IP の導出を数値で確かめる | C | `aux_check.py` |
| 3 | 第7章・pyroomacoustics との照合 | D | `compare_libs.py` |
| 4 | パーミュテーション問題を見て、聞く | E | `permutation_demo.py` |
| 5 | 音源分離の論文の作法をまねた評価（**ノートブックでは種 2 つの試しの設定**） | F | `eval_iv6.py` |
| 6 | 統計の誤用を確かめる | G | `misuse_iv6.py` |
| 7 | 優決定・劣決定 | H | `over_under.py` |
| 8 | 確認問題の答え合わせ | 7 節 | `quiz_check.py` |

- 次のセルたちが、共通ライブラリ（IV-2 の `common/` の `__init__.py`・`stft.py`・`audio_io.py`・`plotting.py`）と、第7章の `bss.py`、この章の `bss6.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章の 6 節と同じなので、ほかのファイルは要りません（第7章の `nmf.py` は、この章のプログラムが使わないので書き出しません）。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックの隣を使う。`bss6.py` の先頭の `sys.path.insert` も同じ）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、日本語のフォントの候補を足すこと、コマンドの引数を `sys.argv` で渡すこと、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」を確かめることだけです。
- **5 節（手順 F）の評価は重い**ので、ノートブックでは試しの設定（`--seeds 2`：残響時間 2 × 種 2 = 4 試行）を既定にしています。教科書の本番（`--seeds 10`、20 試行。執筆時の Mac で約 6 分、Colab ではその数倍）にするときは、セルの `sys.argv` をコメントのとおりに書きかえてください。6 節（手順 G）はこの結果（`out/eval_trace.csv`）を使うので、種 2 つのときは教科書の数値（種 10 通りの平均）とは合いません。ノートブックの確かめは、種の数に関係なく成り立つことだけにしてあります。
- **入れていないもの**：手順 A のうち `setup_iv6.sh`（ファイルを `~/home-lab/` に置く準備。ここでは `%%writefile` が代わり）と `run_iv6.sh`（全部を順に実行するシェルスクリプト。このノートブックが代わり）、手順 I（論文を読む。コードがない）。この章には録音の手順はありません。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav（最大 −10 dBFS）をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。
- 話し声は librosa の例の音源 `libri1`・`libri2`・`libri3` を使います（初回はダウンロードします。ネットワークが要ります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・scipy・scikit-learn の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・scikit-learn・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("sklearn", "scikit-learn"),
                 ("pyroomacoustics", "pyroomacoustics"), ("fast_bss_eval", "fast_bss_eval")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, sklearn, pyroomacoustics, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "scikit-learn", sklearn.__version__,
      "pyroomacoustics", pyroomacoustics.__version__, "librosa", librosa.__version__)
print("（教科書は pyroomacoustics 0.10.1・scipy 1.18.1・scikit-learn 1.9.1 で確かめた）")
import os
for d in ["common", "out"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 6 つのセルで、ほかの章のファイル（IV-2 の `common/`、第7章の `bss.py`）と、この章の共通部品 `bss6.py` を書き出します。

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile bss.py
# 第7章の ~/home-lab/ch7/bss.py（AuxIVA・ILRMA の実装）と同じもの
#!/usr/bin/env python3
"""第7章：AuxIVA と ILRMA を NumPy でゼロから実装する（決定系：音源の数 N ＝ マイクの数 M）

記号（論文の周波数・時間の番号 i, j を、この章では f, t と書く。T はフレームの数で、
      論文の基底行列 T_n（ここでは Tb）とは別のもの）
  X : 観測の STFT。形は (F, T, M)。x_ft = [x_ft1, …, x_ftM]ᵀ
  W : 分離行列。形は (F, N, M)。y_ft = W_f x_ft。W_f の n 行目が w_nfᴴ
  Y : 分離信号 (F, T, N)、P = |Y|²
  r : 音源 n の分散（音源モデル）。AuxIVA は r_nt = √(Σ_f |y_nft|²)（周波数をまとめた大きさ）、
      ILRMA は r_nft = (T_n V_n)_ft（音源ごとの NMF）
更新（反復射影法 IP）
  U_nf = (1/T) Σ_t x_ft x_ftᴴ / r_n(ft)     （AuxIVA では φ(r)=1/r を掛ける）
  w_nf ← (W_f U_nf)⁻¹ e_n、 w_nf ← w_nf / √(w_nfᴴ U_nf w_nf)
コスト（負の対数尤度。反復で増えないことを確かめる）
  ILRMA : Σ_n Σ_ft (|y_nft|²/r_nft + log r_nft) − 2T Σ_f log|det W_f|
  AuxIVA: Σ_n Σ_t r_nt − T Σ_f log|det W_f|      （ラプラス分布の音源モデル）
          log|det| の係数が ILRMA の半分なのは、尺度を変えたラプラス分布の負の対数尤度を2で割ったものだから。
          定数倍なので、反復で増えないことの確認には影響しない
"""
import numpy as np

EPS = 1e-10


def demix(W, X):
    """Y = W X（周波数ごと）"""
    return np.einsum("fnm,ftm->ftn", W, X)


def ip_update(W, X, phi, n):
    """反復射影法で W の n 行目を更新する。phi は (F, T) の重み（1/r）"""
    F, T, M = X.shape
    U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), phi) / T          # (F, M, M)
    WU = W @ U
    e = np.zeros((F, M, 1), dtype=complex)
    e[:, n, 0] = 1.0
    w = np.linalg.solve(WU, e)[:, :, 0]                               # (F, M)
    denom = np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))
    w = w / (denom[:, None] + EPS)
    W[:, n, :] = w.conj()
    return W


def logdet(W):
    return np.sum(np.log(np.abs(np.linalg.det(W)) + EPS))


def projection_back(Y, W, ref=0):
    """スケールのあいまいさを、基準マイク ref で観測した大きさに合わせて直す"""
    A = np.linalg.inv(W)                                              # (F, M, N) 混合行列の推定
    return Y * A[:, ref, :][:, None, :]


def auxiva(X, n_iter=100, return_cost=False):
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    for _ in range(n_iter):
        for n in range(M):
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y[:, :, n]) ** 2, axis=0)) + EPS   # (T,)
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), n)
        if return_cost:
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y) ** 2, axis=0)) + EPS           # (T, N)
            hist.append(float(np.sum(r) - T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)


def ilrma(X, n_basis=4, n_iter=100, seed=0, return_cost=False):
    F, T, M = X.shape
    rng = np.random.default_rng(seed)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    Tb = rng.uniform(0.1, 1.0, (M, F, n_basis))                       # 基底 T_n
    Vb = rng.uniform(0.1, 1.0, (M, n_basis, T))                       # アクティベーション V_n
    R = Tb @ Vb                                                       # (N, F, T)
    hist = []
    for _ in range(n_iter):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)               # (N, F, T)
        # 音源モデルの更新（IS ダイバージェンスの NMF。補助関数法の乗法更新）
        Tb *= np.sqrt(((P / R ** 2) @ np.swapaxes(Vb, 1, 2)) / ((1 / R) @ np.swapaxes(Vb, 1, 2) + EPS))
        R = Tb @ Vb + EPS
        Vb *= np.sqrt((np.swapaxes(Tb, 1, 2) @ (P / R ** 2)) / (np.swapaxes(Tb, 1, 2) @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        # 分離行列の更新（IP）
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
        # スケールの正規化（W の各行と、音源モデルの大きさをそろえる。結果は変わらない）
        Y = demix(W, X)
        lam = np.sqrt(np.mean(np.abs(Y) ** 2, axis=(0, 1))) + EPS     # (N,)
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        if return_cost:
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            hist.append(float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)

In [ ]:
%%writefile bss6.py
#!/usr/bin/env python3
"""IV-6：第7章の bss.py をそのまま使い、理論の確かめに要る道具を足す（この章の共通部品）

第7章の ~/home-lab/ch7/bss.py を import する（demix・ip_update・logdet・projection_back・auxiva・ilrma）。
配列の形は第7章と同じ：X は (F, T, M)、W は (F, N, M)、y_ft = W_f x_ft、Y は (F, T, N)。

この章で足すもの
  sources(names, fs, sec)          : librosa の例の音源を読み、長さと大きさをそろえる
  simulate(src, fs, rt60, seed, …) : pyroomacoustics の鏡像法で M 本のマイクの観測を作る（rt60=0 は無響）
  fdica(X, n_iter)                 : 周波数ごとに独立な ICA（AuxICA）。パーミュテーションは解かない
  align_power_ratio(Y, W)          : パワー比の相関で順番をそろえる（ブラインド。Sawada らの方法の簡易版）
  align_oracle(Y, ref_img)         : 正解を使って順番をそろえる（上限を見るための oracle）
  swapped_stats(perms, energy_db)  : 順番が多数派と違うビンを、エネルギーのあるビンだけで数える
  ilrma_v(X, K, n_iter, …)         : 第7章の ilrma と同じ計算に、初期値・更新の順番・正規化を選ぶ引数を足したもの
  oracle_w(X, img)                 : 正解の像に最小二乗で合わせた分離行列（周波数ごとの線形分離の上限の目安）
  ip_oracle(X, img, n_iter)        : 音源モデルに正解のパワーを入れた IP（音源モデルが完璧なときの上限）
  cost_ilrma / cost_auxiva         : 負の対数尤度（第7章と同じ式）
  to_time / evaluate               : 逆 STFT と BSS Eval（fast_bss_eval）
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ と第7章の bss.py は、このノートブックの隣に書き出してある

import bss                                              # noqa: E402  第7章の実装
from bss import demix, ip_update, logdet, projection_back, EPS   # noqa: E402,F401
from common.stft import stft, istft                     # noqa: E402

L_WIN, S_HOP = 2048, 512                                 # 第IV部の分離の標準（16 kHz で 128 ms、第7章と同じ）


# ---------------------------------------------------------------- 音源と部屋
def sources(names=("libri1", "libri3"), fs=16000, sec=8.0, offset=0.0):
    """librosa の例の音源を読み、同じ長さにして標準偏差を 1 にそろえる。(N, n) を返す"""
    import librosa
    s = [librosa.load(librosa.ex(nm), sr=fs, mono=True, offset=offset, duration=sec)[0] for nm in names]
    n = min(len(x) for x in s)
    return np.stack([x[:n] / (np.std(x[:n]) + 1e-12) for x in s]).astype(float)


def simulate(src, fs, rt60, seed, n_mics=2, d=0.04, dist=1.5, min_sep_deg=30):
    """直方体の部屋（幅 5〜8 m・奥行 4〜6 m・高さ 3 m）に、間隔 d [m] の直線アレイと音源を置く。
    音源はアレイの中心から dist [m]、方向は互いに min_sep_deg 度以上離す（第7章の simulate と同じ考え方）。
    戻り値：mix (M, n)、img (N, n)（各音源を 0 番のマイクで録った像＝BSS Eval の正解）"""
    import pyroomacoustics as pra
    rng = np.random.default_rng(seed)
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
    if rt60 > 0:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
    else:
        room = pra.ShoeBox(dim, fs=fs, max_order=0)        # 反射なし（無響）
    c = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.5])
    off = (np.arange(n_mics) - (n_mics - 1) / 2) * d
    room.add_microphone_array(np.stack([c + [o, 0, 0] for o in off], axis=1))
    N = src.shape[0]
    while True:
        ang = np.sort(rng.uniform(0, np.pi, N))
        if N == 1 or np.min(np.diff(ang)) >= np.radians(min_sep_deg):
            break
    for k in range(N):
        room.add_source(c + dist * np.array([np.cos(ang[k]), np.sin(ang[k]), 0.0]), signal=src[k])
    pre = room.simulate(return_premix=True)                # (音源, マイク, 時間)
    n = src.shape[1]
    return pre.sum(axis=0)[:, :n], pre[:, 0, :n]


def bin_hz(mask, fs=16000):
    """True のビンの周波数の範囲を "a〜b Hz" の文字列で返す（表示用）"""
    f = np.flatnonzero(mask) * fs / L_WIN
    return f"{f.min():.0f}〜{f.max():.0f} Hz"


def to_stft(x):
    """(M, n) の時間信号 → (F, T, M) の STFT（common.stft、L=2048・S=512・hann）"""
    return stft(np.asarray(x).T, L_WIN, S_HOP)


def to_time(Y, length):
    """(F, T, N) → (N, length) の時間信号（common.istft。length は必ず元の長さ）"""
    return np.stack([istft(Y[:, :, n], L_WIN, S_HOP, length=length) for n in range(Y.shape[2])])


def evaluate(ref, est):
    """BSS Eval（fast_bss_eval）。順番は正解に合わせて選ぶ。音源ごとの SDR・SIR・SAR を返す"""
    import fast_bss_eval
    with np.errstate(divide="ignore", invalid="ignore"):
        sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return np.asarray(sdr), np.asarray(sir), np.asarray(sar)


# ---------------------------------------------------------------- コスト（第7章と同じ式）
def cost_ilrma(W, X, R):
    """Σ_n Σ_ft (|y|²/r + log r) − 2T Σ_f log|det W_f|。R は (N, F, T)"""
    T = X.shape[1]
    P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    return float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W))


def cost_auxiva(W, X):
    """第7章の AuxIVA のコスト Σ_n Σ_t r_nt − T Σ_f log|det W_f|（r_nt = √Σ_f |y_nft|²）"""
    T = X.shape[1]
    r = np.sqrt(np.sum(np.abs(demix(W, X)) ** 2, axis=0)) + EPS
    return float(np.sum(r) - T * logdet(W))


# ---------------------------------------------------------------- FDICA とパーミュテーション
def fdica(X, n_iter=100, W0=None, floor=1e-3):
    """周波数ごとに独立な ICA。音源モデルは周波数ごとの複素ラプラス分布 p(y) ∝ exp(−2|y|)、
    つまり AuxIVA の r_nt を r_nft = |y_nft| に変えただけ（周波数どうしをつなぐものが何もない）。
    更新は第7章の ip_update（IP）をそのまま使う。
    |y| が 0 に近いと重み 1/|y| が大きくなりすぎて計算が壊れる（NaN）ので、|y| が floor より小さいところは
    floor に切り上げる。これは「r ≤ floor では 2 乗、それより上では r」のフーバー型の G を使うことにあたり、
    G'(r)/r は増えないので補助関数法が使える。floor は反復の間ずっと同じ定数なので、コスト J も1つに決まり、
    J は反復で増えない。IP の正規化で収束後は平均 |y| ≈ 1 になるので、floor = 1e-3 は「平均の 1000 分の 1」の目安。"""
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1)) if W0 is None else W0.copy()
    for _ in range(n_iter):
        for n in range(M):
            r = np.maximum(np.abs(demix(W, X)[:, :, n]), floor)   # (F, T)
            W = ip_update(W, X, 1.0 / r, n)
    return W


def align_oracle(Y, img_stft):
    """正解の像の STFT img_stft (F, T, N) と、振幅の包絡の相関が最大になる順番を周波数ごとに選ぶ（oracle）"""
    import itertools
    F, T, N = Y.shape
    a = np.abs(Y) - np.abs(Y).mean(axis=1, keepdims=True)
    b = np.abs(img_stft) - np.abs(img_stft).mean(axis=1, keepdims=True)
    a /= np.linalg.norm(a, axis=1, keepdims=True) + 1e-12
    b /= np.linalg.norm(b, axis=1, keepdims=True) + 1e-12
    C = np.einsum("ftn,ftk->fnk", a, b)                    # C[f, 出力 n, 音源 k]
    best = np.zeros((F, N), dtype=int)
    score = np.full(F, -np.inf)
    for p in itertools.permutations(range(N)):             # p[k] = 音源 k に割り当てる出力の番号
        s = sum(C[:, p[k], k] for k in range(N))
        better = s > score
        best[better] = p
        score[better] = s[better]
    return best                                            # Y[:, :, best[f]] が音源の順


def align_power_ratio(Yp, n_rounds=10):
    """ブラインドな順番そろえ：プロジェクションバック後の出力 Yp (F, T, N) のパワー比
    q_nft = |y_nft|² / Σ_m |y_mft|² を「その音源がどれだけ鳴っているか」とみなし、
    全周波数の平均（重心）との相関が最大になる順番を周波数ごとに選ぶ、をくり返す。"""
    import itertools
    F, T, N = Yp.shape
    q = np.abs(Yp) ** 2
    q = q / (q.sum(axis=2, keepdims=True) + 1e-12)
    q = q - q.mean(axis=1, keepdims=True)
    q /= np.linalg.norm(q, axis=1, keepdims=True) + 1e-12
    perms = np.tile(np.arange(N), (F, 1))
    for _ in range(n_rounds):
        cur = np.take_along_axis(q, perms[:, None, :], axis=2)
        cen = cur.mean(axis=0)                             # (T, N) 重心
        C = np.einsum("ftn,tk->fnk", q, cen)
        score = np.full(F, -np.inf)
        new = perms.copy()
        for p in itertools.permutations(range(N)):
            s = sum(C[:, p[k], k] for k in range(N))
            better = s > score
            new[better] = p
            score[better] = s[better]
        if np.array_equal(new, perms):
            break
        perms = new
    return perms


def swapped_stats(perms, energy_db, thr_db=-40.0):
    """2 音源のとき、多いほうの順番と違う周波数を数える。
    正解の像のエネルギーが最大から thr_db より小さいビンは、どちらの順番でも差がほとんどなく
    順番が決まらないので数えない。戻り値：(数えたビンでの割合, 入れ替わったビンの数, エネルギーで重みづけた割合)"""
    p = perms[:, 0]
    use = energy_db > thr_db
    maj = np.bincount(p[use]).argmax()
    sw = (p != maj) & use
    w = 10 ** (energy_db / 10)
    return float(sw.sum() / use.sum()), int(sw.sum()), float(np.sum(w[sw]) / np.sum(w))


def energy_db(img_stft):
    """周波数ごとの正解の像のエネルギー [dB]（最大を 0 dB）"""
    E = np.sum(np.abs(img_stft) ** 2, axis=(1, 2))
    return 10 * np.log10(E / E.max() + 1e-30)


def permute(Y, perms):
    """Y (F, T, N) の出力の順番を周波数ごとに perms (F, N) で並べ替える"""
    return np.take_along_axis(Y, perms[:, None, :], axis=2)


# ---------------------------------------------------------------- ILRMA（第7章と同じ計算＋比べるための引数）
def init_ch7(M, F, T, K, seed):
    """第7章の ilrma と同じ初期値（default_rng(seed) で Tb → Vb の順に一様乱数 0.1〜1.0）"""
    rng = np.random.default_rng(seed)
    return rng.uniform(0.1, 1.0, (M, F, K)), rng.uniform(0.1, 1.0, (M, K, T))


def init_pra(M, F, T, K, seed):
    """pyroomacoustics の ilrma と同じ初期値（np.random.seed(seed) のあと rand で T → V の順。V は転置して返す）"""
    rs = np.random.RandomState(seed)
    Tb = 0.1 + 0.9 * rs.rand(M, F, K)
    Vb = 0.1 + 0.9 * rs.rand(M, T, K)
    return Tb, np.ascontiguousarray(np.swapaxes(Vb, 1, 2))


def ilrma_v(X, n_basis=2, n_iter=100, init=None, seed=0, order="ch7", norm="row", fresh_ir=False,
            return_cost=False, checkpoints=(), on_checkpoint=None):
    """ILRMA。order="ch7" かつ norm="row" かつ init=None なら、第7章の bss.ilrma と同じ計算になる。
      init   : (Tb0, Vb0) を与えるとそれを使う（init_ch7・init_pra で作る）
      order  : "ch7"＝全音源の NMF → 全音源の IP、"pra"＝音源ごとに NMF → IP（pyroomacoustics の順。eps も合わせる）
      fresh_ir : order="pra" のとき、各反復の最初の基底の更新で 1/R を作り直す（True）か、
               pyroomacoustics 0.10.1 と同じく正規化の前の 1/R を使う（False）か
      norm   : "row"＝W の n 行目を割る（正しい）、"col"＝W の n 列目に掛ける（pyroomacoustics 0.10.1 の書き方）、
               None＝正規化しない
      checkpoints と on_checkpoint(i, W) で、途中の反復（i 回目のあと）の W を受け取れる
    戻り値：(Y（プロジェクションバック済み）, W, コストの履歴)"""
    F, T, M = X.shape
    Tb, Vb = init if init is not None else init_ch7(M, F, T, n_basis, seed)
    Tb, Vb = Tb.copy(), Vb.copy()
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    if order == "ch7":
        R = Tb @ Vb
    else:
        eps = 1e-15
        R = np.maximum(Tb @ Vb, eps)
        iR_all = 1 / R                                   # pyroomacoustics は 1/R を別の配列に持つ
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    for it in range(1, n_iter + 1):
        if order == "ch7":
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            Vt = np.swapaxes(Vb, 1, 2)
            Tb *= np.sqrt(((P / R ** 2) @ Vt) / ((1 / R) @ Vt + EPS))
            R = Tb @ Vb + EPS
            Tt = np.swapaxes(Tb, 1, 2)
            Vb *= np.sqrt((Tt @ (P / R ** 2)) / (Tt @ (1 / R) + EPS))
            R = Tb @ Vb + EPS
            for n in range(M):
                W = ip_update(W, X, 1.0 / R[n], n)
        else:                                            # pyroomacoustics の順番と eps の扱い
            for s in range(M):
                iR = 1 / R[s] if fresh_ir else iR_all[s]     # pyroomacoustics は正規化のあとの 1/R を作り直さない
                Tb[s] *= np.sqrt(((P[s] * iR ** 2) @ Vb[s].T) / (iR @ Vb[s].T))
                Tb[Tb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR = iR_all[s] = 1 / R[s]
                Vb[s] *= np.sqrt((Tb[s].T @ (P[s] * iR ** 2)) / (Tb[s].T @ iR))
                Vb[Vb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR_all[s] = 1 / R[s]
                U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), 1 / R[s]) / T
                e = np.zeros((F, M, 1), dtype=complex)
                e[:, s, 0] = 1.0
                w = np.linalg.solve(W @ U, e)[:, :, 0]
                w = w / np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))[:, None]
                W[:, s, :] = w.conj()
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
        if norm is not None:
            Pn = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            lam = np.sqrt(np.mean(Pn, axis=(1, 2))) + (EPS if order == "ch7" else 0.0)   # (N,)
            for s in range(M):
                if norm == "row":
                    W[:, s, :] /= lam[s]
                else:                                    # pyroomacoustics 0.10.1：列に掛けている
                    W[:, :, s] /= lam[s]
                R[s] /= lam[s] ** 2
                Tb[s] /= lam[s] ** 2
                if order != "ch7":
                    P[s] /= lam[s] ** 2
        if return_cost:
            hist.append(cost_ilrma(W, X, R))
        if on_checkpoint is not None and it in checkpoints:
            on_checkpoint(it, W.copy())
    Y = projection_back(demix(W, X), W)
    return Y, W, np.array(hist)


# ---------------------------------------------------------------- oracle（上限の目安）
def oracle_w(X, img_stft):
    """W_f = argmin Σ_t ||s_ft − W x_ft||²（s は正解の像）＝ S Xᴴ (X Xᴴ)⁻¹。出力は最初から像の大きさ"""
    Sxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Ssx = np.einsum("ftn,ftk->fnk", img_stft, X.conj())
    W = np.linalg.solve(np.swapaxes(Sxx, 1, 2), np.swapaxes(Ssx, 1, 2)).swapaxes(1, 2)   # Ssx Sxx⁻¹
    return W


def ip_oracle(X, img_stft, n_iter=20):
    """音源モデル r_nft に正解の像のパワー |s_nft|² を入れて IP だけを回す（音源モデルが完璧な ILRMA）"""
    F, T, M = X.shape
    R = np.moveaxis(np.abs(img_stft) ** 2, 2, 0) + 1e-6 * np.mean(np.abs(img_stft) ** 2)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    for _ in range(n_iter):
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
    return projection_back(demix(W, X), W), W

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
import bss6, bss                                 # bss6 は第7章の bss.py を読み込む
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
print("bss6.py と第7章の bss.py を読み込めました：", bss.__file__)


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.32, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 実数の ICA（IV-6-3-1〜3-4、手順 B、`ica_real.py`）

中心極限定理・白色化・FastICA を確かめます（数十秒）。図は `ica_angle.png`（山が 90° ごと）と `ica_scatter.png`（4 つの散布図）です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-6：実数の ICA を手で確かめる（中心極限定理・白色化・FastICA・自然勾配法・不定性）

使い方: cd ~/home-lab/partIV/iv6 && uv run python ica_real.py
  [1] 混ぜると尖度が 3（正規分布の値）に近づく（中心極限定理の入口）
  [2] 白色化すると、残る未知数は回転だけになる
  [3] 自前の FastICA（対称・logcosh）と自然勾配法を、sklearn.decomposition.FastICA と照合する
  [4] 回転の角度と「非ガウス性」「対数尤度」：分離の角度で山になり、周期 π/2（順番と符号の不定性）
  [5] ガウス分布の音源は分離できない（回転しても分布が同じ）
  [6] 声2つの瞬時混合を FastICA で分け、wav に書き出す（耳で確かめる）
図：out/ica_scatter.png・out/ica_angle.png、音：out/ica_*.wav（最大 −10 dBFS）
"""
import pathlib
import sys

import numpy as np
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する

sys.path.insert(0, ".")   # ノートブック版：common/ と bss6.py はこのノートブックの隣
import common.plotting  # noqa: E402,F401  日本語フォント
from common.audio_io import save  # noqa: E402

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/
OUT.mkdir(exist_ok=True)


def kurt(y):
    """尖度 E[(y−μ)⁴]/σ⁴（正規分布で 3）"""
    y = y - y.mean()
    return np.mean(y ** 4) / np.mean(y ** 2) ** 2


def whiten(x):
    """x (M, n) を白色化する：z = V (x − 平均)、V = D^(−1/2) Eᵀ（C = E D Eᵀ は共分散の固有値分解）"""
    xc = x - x.mean(axis=1, keepdims=True)
    C = xc @ xc.T / xc.shape[1]
    d, E = np.linalg.eigh(C)
    V = np.diag(d ** -0.5) @ E.T
    return V @ xc, V


def sym_decorrelate(W):
    """W ← (W Wᵀ)^(−1/2) W（行どうしを直交させ、長さを 1 にする）"""
    d, E = np.linalg.eigh(W @ W.T)
    return E @ np.diag(d ** -0.5) @ E.T @ W


def fastica(z, n_iter=200, seed=0, tol=1e-12):
    """白色化済みの z (M, n) に対する対称 FastICA（G(u) = log cosh u、g = tanh、g' = 1 − tanh²）
    w ← E[z g(wᵀz)] − E[g'(wᵀz)] w を全部の行で同時に行い、対称な直交化をする"""
    M, n = z.shape
    W = sym_decorrelate(np.random.default_rng(seed).standard_normal((M, M)))
    for it in range(n_iter):
        y = W @ z
        g = np.tanh(y)
        W1 = sym_decorrelate(g @ z.T / n - np.mean(1 - g ** 2, axis=1)[:, None] * W)
        lim = np.max(np.abs(np.abs(np.sum(W1 * W, axis=1)) - 1))   # 向きの変化（符号は無視）
        W = W1
        if lim < tol:
            break
    return W, it + 1


def natgrad_ica(z, n_iter=2000, mu=0.1):
    """自然勾配法：W ← W + μ (I − E[φ(y) yᵀ]) W、φ(y) = tanh(y)（p(y) ∝ 1/cosh(y) のスコア関数）"""
    M, n = z.shape
    W = np.eye(M)
    for _ in range(n_iter):
        y = W @ z
        W = W + mu * (np.eye(M) - np.tanh(y) @ y.T / n) @ W
    return W


def amari_index(P):
    """P = W A が「順番の入れ替え × 対角」にどれだけ近いか（0 なら完全に分離。0〜1 に正規化）"""
    P = np.abs(P)
    M = P.shape[0]
    r = np.sum(np.sum(P, axis=1) / np.max(P, axis=1) - 1)
    c = np.sum(np.sum(P, axis=0) / np.max(P, axis=0) - 1)
    return (r + c) / (2 * M * (M - 1))


def main():
    rng = np.random.default_rng(0)
    n = 200_000
    # ---------------- [1] 中心極限定理：混ぜると正規分布に近づく
    print("[1] 分散 1 のラプラス分布 2 つを y = cosθ s1 + sinθ s2 で混ぜたときの尖度（正規分布で 3）")
    s = rng.laplace(0, 1 / np.sqrt(2), (2, 1_000_000))
    for deg in (0, 22.5, 45):
        th = np.radians(deg)
        y = np.cos(th) * s[0] + np.sin(th) * s[1]
        theory = 3 + 3 * (np.cos(th) ** 4 + np.sin(th) ** 4)
        print(f"  θ = {deg:5.1f}°  乱数 {kurt(y):.3f}  理論 3 + 3(cos⁴θ + sin⁴θ) = {theory:.3f}")
    u = rng.uniform(-np.sqrt(3), np.sqrt(3), (2, 1_000_000))
    print(f"  一様分布（尖度 1.8）を 45° で混ぜる：{kurt((u[0] + u[1]) / np.sqrt(2)):.3f}（理論 3 − 1.2/2 = 2.4）")

    # ---------------- [2] 白色化
    A = np.array([[1.0, 0.6], [0.4, 1.0]])
    s = rng.laplace(0, 1 / np.sqrt(2), (2, n))
    s = (s - s.mean(axis=1, keepdims=True)) / s.std(axis=1, keepdims=True)
    x = A @ s
    z, V = whiten(x)
    Q = V @ A                                                    # z = Q s（s の標本共分散はほぼ I）
    print("\n[2] 白色化")
    print("  z の共分散 =", np.round(z @ z.T / n, 6).tolist())
    print("  Q = V A =", np.round(Q, 4).tolist(), " Q Qᵀ =", np.round(Q @ Q.T, 3).tolist(), "（ほぼ直交 → 残りは回転だけ）")

    # ---------------- [3] FastICA・自然勾配法と sklearn
    from sklearn.decomposition import FastICA
    Wf, its = fastica(z)
    Wn = natgrad_ica(z)
    ica = FastICA(n_components=2, algorithm="parallel", whiten="unit-variance", fun="logcosh",
                  max_iter=1000, tol=1e-12, random_state=0)
    ys = ica.fit_transform(x.T).T
    yf = Wf @ z
    yn = Wn @ z
    print(f"\n[3] FastICA（自前、{its} 回で収束）・自然勾配法・sklearn {ica.n_iter_} 回")
    print(f"  Amari 指数（0 なら完全）：自前 FastICA {amari_index(Wf @ V @ A):.2e}、自然勾配法 {amari_index(Wn @ V @ A):.2e}、"
          f"sklearn {amari_index(ica.components_ @ A):.2e}")
    Cfs = np.corrcoef(np.vstack([yf, ys]))[:2, 2:]
    print("  自前と sklearn の出力の相関（順番と符号は違ってよい）：", np.round(Cfs, 10).tolist())
    print(f"  |相関| の最大と 1 の差：{1 - np.max(np.abs(Cfs), axis=1).min():.1e}（許容 1e-9）")
    Cns = np.corrcoef(np.vstack([yn, ys]))[:2, 2:]
    print(f"  自然勾配法と sklearn：|相関| の最大と 1 の差 {1 - np.max(np.abs(Cns), axis=1).min():.1e}"
          "（不動点の条件が違うので完全には一致しない）")
    print("  sklearn の出力の大きさ（標準偏差）：", np.round(ys.std(axis=1), 4).tolist(), "（白色化で 1。元の大きさは分からない）")

    # ---------------- [4] 回転の角度と非ガウス性・対数尤度
    import matplotlib.pyplot as plt
    ths = np.radians(np.arange(0, 180.5, 0.5))
    J, ll = [], []
    nu = np.mean(np.log(np.cosh(rng.standard_normal(n))))                    # E[G(ν)]、ν は標準正規
    for th in ths:
        R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
        y = R @ z
        J.append(np.sum((np.mean(np.log(np.cosh(y)), axis=1) - nu) ** 2))    # ネゲントロピーの近似
        # 対数尤度（1 サンプルあたり）：log|det W| + Σ_n E[log q(y_n)]、q はラプラス（分散 1）
        ll.append(np.log(abs(np.linalg.det(R @ V))) + np.sum(np.mean(-np.sqrt(2) * np.abs(y) - np.log(np.sqrt(2)), axis=1)))
    J, ll = np.array(J), np.array(ll)
    best = np.degrees(ths[np.argmax(ll)])
    print(f"\n[4] 対数尤度が最大の角度 {best:.1f}°、"
          f"山の角度（ネゲントロピー）{np.degrees(ths[np.argmax(J)]):.1f}°。山は 90° ごと（順番と符号の不定性）")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(np.degrees(ths), J)
    ax[0].set_xlabel("回転の角度 θ [度]")
    ax[0].set_ylabel("非ガウス性 (E log cosh y − E log cosh ν)² の和")
    ax[1].plot(np.degrees(ths), ll, color="C1")
    ax[1].set_xlabel("回転の角度 θ [度]")
    ax[1].set_ylabel("対数尤度 / サンプル（ラプラスのモデル）")
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "ica_angle.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する

    # 散布図：音源・観測・白色化・分離
    fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
    k = rng.choice(n, 4000, replace=False)
    for a, d, t in zip(ax, (s, x, z, ys), ("音源 s（ラプラス）", "観測 x = A s", "白色化 z = V x", "分離 y（sklearn）")):
        a.plot(d[0, k], d[1, k], ".", ms=1.5, alpha=0.5)
        a.set_title(t)
        a.set_aspect("equal")
        a.set_xlim(-6, 6)
        a.set_ylim(-6, 6)
    fig.tight_layout()
    fig.savefig(OUT / "ica_scatter.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する

    # ---------------- [5] ガウス分布の音源
    g = rng.standard_normal((2, n))
    zg, Vg = whiten(A @ g)
    print("\n[5] ガウス分布の音源（同じ A）：初期値を変えると答えが変わり、Amari 指数も大きい")
    for sd in range(3):
        Wg, _ = fastica(zg, seed=sd, n_iter=200)
        print(f"  初期値 {sd}：Amari 指数 {amari_index(Wg @ Vg @ A):.3f}、出力の尖度 {[round(float(kurt(r)), 3) for r in Wg @ zg]}")

    # ---------------- [6] 声2つの瞬時混合を分ける
    try:
        import librosa
        sp = [librosa.load(librosa.ex(nm), sr=16000, duration=8.0)[0] for nm in ("libri1", "libri3")]
        m = min(len(v) for v in sp)
        sp = np.stack([v[:m] / np.std(v[:m]) for v in sp])
    except Exception as e:                                       # ダウンロードできないとき
        print("  librosa の例の音源が読めないので、ラプラス雑音で代用します：", e)
        sp = rng.laplace(0, 1, (2, 16000 * 8))
    xs = np.array([[1.0, 0.6], [0.5, 1.0]]) @ sp
    ica2 = FastICA(n_components=2, whiten="unit-variance", random_state=0, max_iter=1000)
    ysp = ica2.fit_transform(xs.T).T
    C = np.corrcoef(np.vstack([ysp, sp]))[:2, 2:]
    print("\n[6] 声の瞬時混合：出力と音源の相関（行が出力、列が音源）", np.round(C, 3).tolist())
    print(f"  尖度：音源 {[round(float(kurt(v)), 2) for v in sp]}、混合 {[round(float(kurt(v)), 2) for v in xs]}、分離 {[round(float(kurt(v)), 2) for v in ysp]}")
    pk = 10 ** (-10 / 20)
    for name, sig in (("ica_mix1", xs[0]), ("ica_mix2", xs[1]), ("ica_out1", ysp[0]), ("ica_out2", ysp[1])):
        save(OUT / f"{name}.wav", sig / np.max(np.abs(sig)) * pk, 16000)
    print("  out/ica_mix1.wav・ica_mix2.wav（混合）、ica_out1.wav・ica_out2.wav（分離）を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
k = re.findall(r"乱数 ([\d.]+)  理論 .*? = ([\d.]+)", cap.stdout)
ok([float(b) for a, b in k] == [6.0, 5.25, 4.5] and all(abs(float(a) - float(b)) <= 0.02 for a, b in k),
   "[1] 尖度が 6.0・5.25・4.5（理論との差 0.02 以内）: " + str(k))
qq = re.search(r"Q Qᵀ = (\[\[.*?\]\])", cap.stdout).group(1)
ok(qq.replace("-0.0", "0.0") == "[[1.0, 0.0], [0.0, 1.0]]", "[2] Q Qᵀ が単位行列 " + qq)
ok(num(r"\|相関\| の最大と 1 の差：([\d.e+-]+)（許容") < 1e-9, "[3] 「|相関| の最大と 1 の差」が 1e-9 より小さい（教科書 4.4e-16）")
a1, a2 = num(r"最大の角度 ([\d.]+)°"), num(r"山の角度（ネゲントロピー）([\d.]+)°")
ok(a1 == a2, f"[4] 山の角度が2つの方法で同じ（{a1}°。教科書 42.0°）")
am = [float(v) for v in re.findall(r"初期値 \d：Amari 指数 ([\d.]+)", cap.stdout)]
ok(len(set(am)) > 1, f"[5] ガウス分布の音源では、初期値によって Amari 指数が変わる {am}")
C = eval(re.search(r"列が音源） (\[\[.*?\]\])", cap.stdout).group(1))
ok(all(sorted(abs(v) for v in row)[-1] > 0.95 and sorted(abs(v) for v in row)[0] < 0.1 for row in C)
   and {max(range(2), key=lambda j: abs(row[j])) for row in C} == {0, 1},
   f"[6] 相関の表で、各行に絶対値が 1.0 に近い値が1つずつある {C}")

混合と分離を聞き比べます（手順 B の 3）。混合では2人の声が重なり、分離ではほぼ1人ずつになります。出力の順番（どちらが男性の声か）は実行によって決まり、指定できません（3-4）。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['out/ica_mix1.wav', 'out/ica_out1.wav', 'out/ica_out2.wav']:
    print(f)
    listen_file(f)

## 2. 補助関数と IP の導出を数値で確かめる（IV-6-3-7〜3-9、手順 C、`aux_check.py`）

数秒です。[3] の数値は確認問題4（2×2 の IP）の答えと同じです。**先に紙で解いてから**出力と比べてください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-6：AuxIVA・ILRMA の導出を数値で確かめる

使い方: cd ~/home-lab/partIV/iv6 && uv run python aux_check.py
  [1] 補助関数の不等式 G(r) ≤ G'(r0)/(2 r0)·r² + 定数（G'(r)/r が増えないときだけ成り立つ）
  [2] IP の更新は「1 行についての大域的な最小」（ランダムな 2 万個の行より小さい）と、停留条件 w_mᴴ V_n w_n = δ_mn
  [3] 2×2 の手計算の例（確認問題 4 と同じ数）
  [4] AuxIVA：1 行更新するごとに、本当のコスト J が増えない（第7章の ip_update を使う）
  [5] ILRMA：基底・係数・各行の IP・正規化の 4 段階のどれでも J が増えない。正規化で J は変わらない
  [6] ガウス分布の音源モデル（重みが一定）では、白色化までしかできない
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ と bss6.py はこのノートブックの隣
import bss6                                         # noqa: E402  （第7章の bss.py も読み込まれる）
from bss import demix, ip_update, logdet, EPS      # noqa: E402

rng = np.random.default_rng(0)


def crandn(*shape):
    return (rng.standard_normal(shape) + 1j * rng.standard_normal(shape)) / np.sqrt(2)


def rand_psd(M):
    B = crandn(M, 3 * M)
    return B @ B.conj().T / (3 * M)


def Q(W, V):
    """1 つの周波数の補助関数（の T で割ったもの）：Σ_n w_nᴴ V_n w_n − 2 log|det W|。W の n 行目が w_nᴴ"""
    quad = sum(np.real(W[n] @ V[n] @ W[n].conj()) for n in range(W.shape[0]))
    return float(quad - 2 * np.log(abs(np.linalg.det(W))))


def ip_row(W, Vn, n):
    """w_n ← (W V_n)⁻¹ e_n、w_n ← w_n / √(w_nᴴ V_n w_n)。W の n 行目に w_nᴴ を入れて返す"""
    M = W.shape[0]
    w = np.linalg.solve(W @ Vn, np.eye(M)[:, n])
    w = w / np.sqrt(np.real(w.conj() @ Vn @ w))
    W = W.copy()
    W[n] = w.conj()
    return W


def main():
    # ---------------- [1] 補助関数の不等式
    print("[1] G(r) ≤ G'(r0)/(2r0)·r² + G(r0) − r0·G'(r0)/2（r0 = 1.3、r は 0〜10 の 10 万点）")
    r = np.linspace(1e-6, 10, 100_000)
    r0 = 1.3
    for name, G, dG in (("ラプラス G=r", lambda r: r, lambda r: np.ones_like(r)),
                        ("一般化ガウス G=r^0.5", lambda r: r ** 0.5, lambda r: 0.5 * r ** -0.5),
                        ("ガウス G=r²", lambda r: r ** 2, lambda r: 2 * r),
                        ("G=r³（G'(r)/r が増える）", lambda r: r ** 3, lambda r: 3 * r ** 2)):
        g0 = dG(np.array(r0)) / (2 * r0)
        bound = g0 * r ** 2 + G(r0) - r0 * dG(np.array(r0)) / 2
        gap = bound - G(r)
        print(f"  {name:24s} 上界 − G の最小 {gap.min():+.3e}（0 以上なら成立）、r0 での差 {float(g0 * r0**2 + G(r0) - r0 * dG(np.array(r0)) / 2 - G(r0)):+.1e}")

    # ---------------- [2] IP は 1 行についての大域的な最小
    M = 3
    W = crandn(M, M)
    V = [rand_psd(M) for _ in range(M)]
    n = 1
    W1 = ip_row(W, V[n], n)
    q0, q1 = Q(W, V), Q(W1, V)
    worst = np.inf
    for _ in range(20000):
        Wr = W1.copy()
        Wr[n] = W1[n] + 0.5 * crandn(M) * rng.uniform(0.001, 1)
        worst = min(worst, Q(Wr, V) - q1)
    # コーシー–シュワルツから出る最小値：1 − log(bᴴ V⁻¹ b) + （他の行の項）
    b = np.linalg.inv(W)[:, n] * np.linalg.det(W)                 # 余因子のベクトル（n 行目によらない）
    others = sum(np.real(W[m] @ V[m] @ W[m].conj()) for m in range(M) if m != n)
    qmin = 1 - np.log(np.real(b.conj() @ np.linalg.solve(V[n], b))) + others
    print(f"\n[2] 3×3 の例：更新前 Q = {q0:.6f} → IP 後 Q = {q1:.6f}（理論の最小 {qmin:.6f}、差 {abs(q1 - qmin):.1e}）")
    print(f"  n 行目をランダムに動かした 2 万個の Q − IP 後の Q の最小 {worst:+.3e}（正なら IP が最小）")
    # すべての行を何周か IP で回したあとの停留条件
    Wc = W.copy()
    for _ in range(200):
        for m in range(M):
            Wc = ip_row(Wc, V[m], m)
    Hm = np.array([[Wc[m] @ V[k] @ Wc[k].conj() for k in range(M)] for m in range(M)])   # [m, k] = w_mᴴ V_k w_k
    print("  200 周後の w_mᴴ V_k w_k（単位行列になるはず）の |単位行列との差| の最大",
          f"{np.max(np.abs(Hm - np.eye(M))):.1e}")

    # ---------------- [3] 2×2 の手計算
    print("\n[3] 2×2・実数の例：W = I、V_1 = [[2,1],[1,2]]、V_2 = [[1,0],[0,4]]")
    W = np.eye(2, dtype=complex)
    V1 = np.array([[2, 1], [1, 2]], dtype=complex)
    V2 = np.array([[1, 0], [0, 4]], dtype=complex)
    print(f"  Q（更新前）= {Q(W, [V1, V2]):.6f}")
    w = np.linalg.solve(W @ V1, [1, 0])
    print(f"  1 行目：(W V_1)⁻¹ e_1 = {np.real(w).round(6).tolist()}、wᴴ V_1 w = {np.real(w.conj() @ V1 @ w):.6f}")
    W = ip_row(W, V1, 0)
    print(f"  正規化後の w_1 = {np.real(W[0]).round(6).tolist()}、Q = {Q(W, [V1, V2]):.6f}")
    w = np.linalg.solve(W @ V2, [0, 1])
    print(f"  2 行目：(W V_2)⁻¹ e_2 = {np.real(w).round(6).tolist()}、wᴴ V_2 w = {np.real(w.conj() @ V2 @ w):.6f}")
    W = ip_row(W, V2, 1)
    print(f"  正規化後の w_2 = {np.real(W[1]).round(6).tolist()}、Q = {Q(W, [V1, V2]):.6f}")
    print(f"  確かめ：w_1ᴴ V_2 w_2 = {np.real(W[0] @ V2 @ W[1].conj()):.2e}（0 になる）、"
          f"w_2ᴴ V_1 w_1 = {np.real(W[1] @ V1 @ W[0].conj()):.4f}（1 行目の条件は、2 行目を変えたので崩れる）")

    # ---------------- 実際の混合音で [4]・[5]
    src = bss6.sources(sec=4.0)
    mix, img = bss6.simulate(src, 16000, 0.15, seed=0)
    X = bss6.to_stft(mix)
    F, T, M = X.shape

    # ---------------- [4] AuxIVA：行ごとに J が増えない
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = [bss6.cost_auxiva(W, X)]
    for it in range(30):
        for n in range(M):
            rr = np.sqrt(np.sum(np.abs(demix(W, X)[:, :, n]) ** 2, axis=0)) + EPS
            W = ip_update(W, X, np.broadcast_to(1.0 / rr, (F, T)), n)
            hist.append(bss6.cost_auxiva(W, X))
    d = np.diff(hist)
    print(f"\n[4] AuxIVA（第7章の ip_update、4 秒の音、30 周 × {M} 行）：J {hist[0]:.1f} → {hist[-1]:.1f}、"
          f"1 行ごとの変化の最大 {d.max():+.2e}（0 以下なら増えていない）")
    # 補助関数 ≥ J をランダムな W で確かめる
    Wa = W.copy()
    Yh = demix(Wa, X)
    rh = np.sqrt(np.sum(np.abs(Yh) ** 2, axis=0)) + EPS             # (T, N) 今の値 r̂
    gaps = []
    for _ in range(200):
        Wr = Wa * (1 + 0.2 * crandn(F, M, M))                     # 今の W のまわりを 20% ほど動かす
        Yr = demix(Wr, X)
        rr = np.sqrt(np.sum(np.abs(Yr) ** 2, axis=0)) + EPS
        J = np.sum(rr) - T * logdet(Wr)
        G = np.sum(rr ** 2 / (2 * rh) + rh / 2) - T * logdet(Wr)      # r ≤ r²/(2r̂) + r̂/2
        gaps.append((G - J) / abs(J))
    print(f"  (補助関数 − J)/|J|（今の W のまわりのランダムな W 200 個）の最小 {min(gaps):+.3e}（0 以上）")

    # ---------------- [5] ILRMA：4 段階すべてで J が増えない
    K = 2
    Tb, Vb = bss6.init_ch7(M, F, T, K, seed=0)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    R = Tb @ Vb
    steps = {"基底 T": [], "係数 V": [], "IP（各行）": [], "正規化": []}
    J0 = bss6.cost_ilrma(W, X, R)
    first = J0
    for it in range(30):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
        Vt = np.swapaxes(Vb, 1, 2)
        Tb *= np.sqrt(((P / R ** 2) @ Vt) / ((1 / R) @ Vt + EPS))
        R = Tb @ Vb + EPS
        J1 = bss6.cost_ilrma(W, X, R)
        steps["基底 T"].append(J1 - J0)
        Tt = np.swapaxes(Tb, 1, 2)
        Vb *= np.sqrt((Tt @ (P / R ** 2)) / (Tt @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        J2 = bss6.cost_ilrma(W, X, R)
        steps["係数 V"].append(J2 - J1)
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
            J3 = bss6.cost_ilrma(W, X, R)
            steps["IP（各行）"].append(J3 - J2)
            J2 = J3
        lam = np.sqrt(np.mean(np.abs(demix(W, X)) ** 2, axis=(0, 1)))
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        J0 = bss6.cost_ilrma(W, X, R)
        steps["正規化"].append(J0 - J3)
    print(f"\n[5] ILRMA（K = 2、30 回）：J {first:.1f} → {J0:.1f}")
    for k, v in steps.items():
        print(f"  {k:10s} 変化の最大 {max(v):+.2e}、合計 {sum(v):+.1f}")
    # 正規化を「列」に掛ける書き方（pyroomacoustics 0.10.1）では J が変わる
    Wc = W.copy()
    lam = np.array([1.7, 0.6])
    Wr, Rr = Wc / lam[None, :, None], R / lam[:, None, None] ** 2
    Wl = Wc / lam[None, None, :]
    print(f"  わざと λ = (1.7, 0.6) で割る：行を割ると J の変化 {bss6.cost_ilrma(Wr, X, Rr) - bss6.cost_ilrma(Wc, X, R):+.2e}、"
          f"列を割ると {bss6.cost_ilrma(Wl, X, Rr) - bss6.cost_ilrma(Wc, X, R):+.3e}")

    # ---------------- [6] ガウス分布の音源モデル
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    for _ in range(5):
        for n in range(M):
            W = ip_update(W, X, np.ones((F, T)), n)                 # 重み 1/r が一定
    Cx = np.einsum("ftm,ftk->fmk", X, X.conj()) / T
    WCW = W @ Cx @ np.conj(np.swapaxes(W, 1, 2))
    th = 0.7
    Rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    W2 = Rot[None] @ W                                              # 回転させた別の W
    def J(W):                                                       # Σ_f [Σ_n w_nᴴ C w_n − 2 log|det W_f|]
        return float(np.real(np.einsum("fnm,fmk,fnk->", W, Cx, W.conj())) - 2 * logdet(W))
    print(f"\n[6] ガウス分布の音源モデル：W C Wᴴ と単位行列の差の最大 {np.max(np.abs(WCW - np.eye(M))):.1e}"
          f"（白色化になる）。回転した W とのコストの差 {J(W2) - J(W):+.1e}（同じ値 → どれが正解か決まらない）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
gaps = {m.group(1): float(m.group(2)) for m in re.finditer(r"(G=r\S*)\s.*?上界 − G の最小 ([-+\d.e]+)", cap.stdout)}
ok(all(gaps[g] > -1e-12 for g in ("G=r", "G=r^0.5", "G=r²")) and gaps["G=r³（G'(r)/r"] < -1, f"[1] G = r・r^0.5・r² は差が 0 以上（丸め誤差を除く）、G = r³ だけ負 {gaps}")
ok(num(r"理論の最小 [\d.]+、差 ([\d.e+-]+)）") < 1e-12 and num(r"2 万個の Q − IP 後の Q の最小 ([-+\d.e]+)") > 0,
   "[2] IP の後の値が理論の最小と 1e-12 以内で一致し、ランダムな行がすべてそれより大きい")
ok("正規化後の w_1 = [0.816497, -0.408248]、Q = 5.405465" in cap.stdout and "正規化後の w_2 = [0.707107, 0.353553]、Q = 3.098612" in cap.stdout
   and "Q（更新前）= 6.000000" in cap.stdout,
   "[3] 確認問題4の答えと同じ：w_1 = [0.8165, −0.4082]、w_2 = [0.7071, 0.3536]、q が 6 → 5.4055 → 3.0986")
ok(num(r"1 行ごとの変化の最大 ([-+\d.e]+)") < 0, "[4] AuxIVA の 1 行ごとの変化の最大が負")
ch = {m.group(1): float(m.group(2)) for m in re.finditer(r"^  (基底 T|係数 V|IP（各行）|正規化)\s+変化の最大 ([-+\d.e]+)", cap.stdout, re.M)}
ok(len(ch) == 4 and all(v < 0 for k, v in ch.items() if k != "正規化") and abs(ch["正規化"]) < 1e-3,
   f"[5] ILRMA の各更新の変化の最大が負（正規化は 1e-5 程度の丸め誤差）{ch}")

## 3. 第7章・pyroomacoustics と照合する（IV-6-3-6・3-9・3-10、手順 D、`compare_libs.py`）

1〜2 分かかります（教科書の目安は 2〜4 分）。図 `compare_cost.png` で、3本の線がほぼ重なって下がること、pra そのまま（列に掛ける）の線にだけ、わずかに上がる回があることを見てください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['compare_libs.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-6：第7章の AuxIVA・ILRMA を pyroomacoustics と数値で照合する（同じ初期値・同じ反復数）

使い方: cd ~/home-lab/partIV/iv6 && uv run python compare_libs.py [--iter 100] [--seeds 3]
  [1] AuxIVA（ラプラス）：第7章と pra.bss.auxiva。W の行の大きさは 2 倍違うが、プロジェクションバック後は一致する
  [2] AuxIVA（時間で変わる分散のガウス）：IP を自分で回したものと pra.bss.auxiva(model="gauss")
  [3] ILRMA：第7章 = bss6.ilrma_v（同じ計算か）、pra.bss.ilrma = bss6.ilrma_v(order="pra", norm="col")（同じ計算か）
      → 違い（更新の順番・正規化の書き方・初期値の乱数・プロジェクションバック）が、コストと SDR にどう効くか
図：out/compare_cost.png。結果の表：out/compare_libs.csv
"""
import argparse
import pathlib
import time

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import pyroomacoustics as pra

import bss6
import common.plotting  # noqa: F401  日本語フォント（bss6 が partIV を sys.path に入れている）
from bss import demix, ip_update, projection_back, auxiva, ilrma, EPS

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/
OUT.mkdir(exist_ok=True)


def rel(a, b):
    return float(np.max(np.abs(a - b)) / np.max(np.abs(b)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--rt60", type=float, default=0.15)
    a = ap.parse_args()
    src = bss6.sources(sec=8.0)
    mix, img = bss6.simulate(src, 16000, a.rt60, seed=0)
    n = mix.shape[1]
    X = bss6.to_stft(mix)
    Xp = X.transpose(1, 0, 2).copy()                         # pyroomacoustics の並び (T, F, M)
    F, T, M = X.shape
    print(f"音：libri1 と libri3（8 秒）、RT60 {a.rt60} s、X の形 {X.shape}、反復 {a.iter} 回\n")
    sdr_of = lambda Y: float(np.mean(bss6.evaluate(img, bss6.to_time(Y, n))[0]))
    rows = []

    # ---------------- [1] AuxIVA（ラプラス）
    t0 = time.time(); Y7, W7 = auxiva(X, a.iter); t7 = time.time() - t0
    t0 = time.time(); Yq, Wq = pra.bss.auxiva(Xp, n_iter=a.iter, proj_back=False, return_filters=True); tq = time.time() - t0
    Yq_pb = projection_back(demix(Wq, X), Wq)                # 第7章と同じプロジェクションバック（W⁻¹）
    Yq_ls = pra.bss.auxiva(Xp, n_iter=a.iter).transpose(1, 0, 2)   # pra の既定（最小二乗のプロジェクションバック）
    print("[1] AuxIVA（ラプラス、初期値はどちらも単位行列）")
    print(f"  W の行の大きさの比 |W_pra / W_第7章| の中央値 {np.median(np.abs(Wq / W7)):.6f}（理論 2）")
    print(f"  同じプロジェクションバックをしたあとの Y の相対差 {rel(Yq_pb, Y7):.1e}（許容 1e-8）")
    s7, sq = sdr_of(Y7), sdr_of(Yq_ls)
    print(f"  SDR：第7章 {s7:.3f} dB（W⁻¹ で戻す）、pra の既定 {sq:.3f} dB（最小二乗で戻す）。時間 {t7:.1f} s と {tq:.1f} s")
    rows += [dict(part="AuxIVA", variant="第7章（W⁻¹ の PB）", sdr=s7, sec=t7),
             dict(part="AuxIVA", variant="pra（最小二乗の PB）", sdr=sq, sec=tq)]

    # ---------------- [2] AuxIVA（ガウス、時間で変わる分散）
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    for _ in range(a.iter):
        for k in range(M):
            r = np.mean(np.abs(demix(W, X)[:, :, k]) ** 2, axis=0) + EPS   # r_kt = (1/F) Σ_f |y|²
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), k)
    Yg = projection_back(demix(W, X), W)
    Yg_q, Wg_q = pra.bss.auxiva(Xp, n_iter=a.iter, model="gauss", proj_back=False, return_filters=True)
    print("\n[2] AuxIVA（ガウス、r_nt = (1/F)Σ_f|y_nft|²）")
    print(f"  自分で回した IP と pra(model='gauss')：プロジェクションバック後の相対差 "
          f"{rel(projection_back(demix(Wg_q, X), Wg_q), Yg):.1e}、SDR {sdr_of(Yg):.3f} dB")
    rows.append(dict(part="AuxIVA-gauss", variant="自前 IP", sdr=sdr_of(Yg), sec=np.nan))

    # ---------------- [3] ILRMA
    print("\n[3] ILRMA（K = 2）")
    Y1, W1 = ilrma(X, 2, 10, seed=0)
    _, W2, _ = bss6.ilrma_v(X, 2, 10, seed=0)
    print(f"  第7章の ilrma と bss6.ilrma_v（10 回）：W の差の最大 {np.max(np.abs(W1 - W2)):.1e}（同じ計算）")
    np.random.seed(0)
    _, Wp = pra.bss.ilrma(Xp, n_iter=10, n_components=2, proj_back=False, return_filters=True)
    _, Wv, _ = bss6.ilrma_v(X, 2, 10, init=bss6.init_pra(M, F, T, 2, 0), order="pra", norm="col")
    print(f"  pra.bss.ilrma と ilrma_v(order='pra', norm='col')（10 回）：W の相対差 {rel(Wv, Wp):.1e}（許容 1e-8）")

    variants = [("第7章の順番・行を正規化", dict(order="ch7", norm="row")),
                ("pra の順番・行を正規化", dict(order="pra", norm="row", fresh_ir=True)),
                ("pra そのまま（列に掛ける）", dict(order="pra", norm="col"))]
    hists = {}
    for seed in range(a.seeds):
        init = bss6.init_pra(M, F, T, 2, seed)                # 3 つとも同じ初期値（pra の乱数）
        for name, kw in variants:
            t0 = time.time()
            Y, W, h = bss6.ilrma_v(X, 2, a.iter, init=init, return_cost=True, **kw)
            sec = time.time() - t0
            s = sdr_of(Y)
            ups = int(np.sum(np.diff(h) > 1e-6 * np.abs(h[1:])))
            rows.append(dict(part="ILRMA", variant=name, seed=seed, sdr=s, cost=h[-1], n_up=ups, sec=sec))
            hists[(seed, name)] = h
            print(f"  初期値 {seed}  {name:18s} SDR {s:6.2f} dB  最後の J {h[-1]:.1f}  J が増えた回 {ups}  {sec:5.1f} s")
        np.random.seed(seed)
        t0 = time.time()
        Yp = pra.bss.ilrma(Xp, n_iter=a.iter, n_components=2).transpose(1, 0, 2)
        sec = time.time() - t0
        rows.append(dict(part="ILRMA", variant="pra（最小二乗の PB）", seed=seed, sdr=sdr_of(Yp), sec=sec))
        print(f"  初期値 {seed}  {'pra の既定（PB も pra）':18s} SDR {sdr_of(Yp):6.2f} dB  {sec:5.1f} s")
    for seed in range(a.seeds):                              # 第7章の関数そのもの（初期値の乱数は第7章の作り方）
        s = sdr_of(ilrma(X, 2, a.iter, seed=seed)[0])
        rows.append(dict(part="ILRMA", variant="第7章そのまま（第7章の乱数）", seed=seed, sdr=s))
        print(f"  初期値 {seed}  {'第7章そのまま':18s} SDR {s:6.2f} dB（初期値の乱数の作り方が違う）")
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "compare_libs.csv", index=False)
    print("\n手法ごとの SDR（初期値の平均）")
    print(df[df.part == "ILRMA"].groupby("variant")["sdr"].agg(["mean", "std", "count"]).round(3).to_string())

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    jmin = min(h.min() for h in hists.values())
    for (seed, name), h in hists.items():
        if seed == 0:
            ax.semilogy(np.arange(1, len(h) + 1), h - jmin + 1, label=name)
    ax.set_xlabel("反復の回数")
    ax.set_ylabel("J − (いちばん小さい J) + 1")
    ax.set_title("ILRMA のコストの推移（初期値 0、3 つとも同じ初期値）")
    ax.legend()
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(OUT / "compare_cost.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("out/compare_cost.png と out/compare_libs.csv を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"\|W_pra / W_第7章\| の中央値 ([\d.]+)") == 2.0 and num(r"同じプロジェクションバックをしたあとの Y の相対差 ([\d.e+-]+)") < 1e-8,
   "[1] 行の大きさの比 2.000000、同じプロジェクションバック後の相対差が 1e-8 未満")
ok(num(r"pra\(model='gauss'\)：プロジェクションバック後の相対差 ([\d.e+-]+)") < 1e-8, "[2] 相対差が 1e-8 未満")
ok(num(r"bss6\.ilrma_v（10 回）：W の差の最大 ([\d.e+-]+)") == 0 and num(r"norm='col'\)（10 回）：W の相対差 ([\d.e+-]+)") < 1e-8,
   "[3] 第7章の ilrma と bss6.ilrma_v の差 0、pra.bss.ilrma と ilrma_v(order='pra', norm='col') の相対差が 1e-8 未満")
tab = {mm.group(1).strip(): float(mm.group(2)) for mm in re.finditer(r"^(\S.*?)\s+([\d.]+)\s+[\d.]+\s+\d+$", cap.stdout.split("手法ごとの SDR")[1], re.M)}
low = tab.pop("pra（最小二乗の PB）")
ok(all(1 < v - low < 3 for v in tab.values()), f"SDR の表で、pra（最小二乗の PB）だけが 2 dB ほど低い（{low} と {tab}）")
inc = [int(v) for v in re.findall(r"pra そのまま（列に掛ける）\s+SDR\s+[\d.]+ dB\s+最後の J [\d.]+\s+J が増えた回 (\d+)", cap.stdout)]
other = [int(v) for v in re.findall(r"行を正規化\s+SDR\s+[\d.]+ dB\s+最後の J [\d.]+\s+J が増えた回 (\d+)", cap.stdout)]
ok(all(v == 0 for v in other), f"行を正規化した版は J が増えた回が 0（pra そのまま（列に掛ける）は {inc}。0 でないことがある）")

## 4. パーミュテーション問題を見て・聞く（IV-6-3-5、手順 E、`permutation_demo.py`）

1 分ほどです。図 `perm_map.png` の上の帯で、FDICA の色が細かく入れ替わること、AuxIVA の低い周波数（62〜141 Hz）に数本の入れ替わりがあることを見てください。灰色はエネルギーの小さいビンで、数えません。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['permutation_demo.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-6：パーミュテーション問題をわざと起こして、見て・聞く

使い方: cd ~/home-lab/partIV/iv6 && uv run python permutation_demo.py [--rt60 0.15] [--seed 0]
  FDICA（周波数ごとに独立な ICA）で分離すると、周波数ごとに出力の順番がばらばらになる。
  それを (a) そのまま、(b) パワー比の相関でそろえる（ブラインド）、(c) 正解でそろえる（oracle）の 3 通りで聞き、
  周波数をつなぐ音源モデルをもつ AuxIVA・ILRMA（第7章）と比べる。
  入れ替わりは、正解の像のエネルギーが最大から −40 dB 以内のビンだけで数える（音のないビンは順番が決まらない）。
  「正解で並べ替えた SDR」と元の SDR の差が、その方法でパーミュテーションが失った分の目安になる。
図：out/perm_map.png。音：out/perm_*.wav（最大 −10 dBFS。再生前に音量を小さく）
"""
import argparse
import pathlib
import time

import numpy as np
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する

import fast_bss_eval

import bss6
import common.plotting  # noqa: F401
from common.plotting import specshow
from common.audio_io import save
from bss import demix, projection_back, auxiva, ilrma

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/
OUT.mkdir(exist_ok=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rt60", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--iter", type=int, default=100)
    a = ap.parse_args()
    fs = 16000
    src = bss6.sources(sec=8.0)
    mix, img = bss6.simulate(src, fs, a.rt60, a.seed)
    n = mix.shape[1]
    X = bss6.to_stft(mix)
    S = bss6.to_stft(img)                                     # 正解の像の STFT (F, T, N)
    F = X.shape[0]

    res = {}
    t0 = time.time()
    Wf = bss6.fdica(X, a.iter)
    Yraw = projection_back(demix(Wf, X), Wf)
    print(f"FDICA {a.iter} 回：{time.time() - t0:.1f} s")
    p_or = bss6.align_oracle(Yraw, S)
    p_pr = bss6.align_power_ratio(Yraw)
    res["FDICA（そろえない）"] = (Yraw, p_or)
    Ypr = bss6.permute(Yraw, p_pr)
    res["FDICA＋パワー比でそろえる"] = (Ypr, bss6.align_oracle(Ypr, S))
    Yor = bss6.permute(Yraw, p_or)
    res["FDICA＋正解でそろえる（oracle）"] = (Yor, bss6.align_oracle(Yor, S))
    t0 = time.time()
    Ya, _ = auxiva(X, a.iter)
    print(f"AuxIVA {a.iter} 回：{time.time() - t0:.1f} s")
    res["AuxIVA"] = (Ya, bss6.align_oracle(Ya, S))
    t0 = time.time()
    Yi, _ = ilrma(X, 2, a.iter, seed=a.seed)
    print(f"ILRMA {a.iter} 回：{time.time() - t0:.1f} s")
    res["ILRMA（K=2）"] = (Yi, bss6.align_oracle(Yi, S))

    sdr_in = bss6.evaluate(img, np.stack([mix[0], mix[0]]))[0]
    E = np.sum(np.abs(S) ** 2, axis=(1, 2))                   # 周波数ごとの正解の像のエネルギー
    Edb = 10 * np.log10(E / E.max() + 1e-30)
    use = Edb > -40
    print(f"\nRT60 {a.rt60} s、seed {a.seed}。混合音の SDR {np.mean(sdr_in):.2f} dB。"
          f"エネルギーが最大から −40 dB 以内のビン {use.sum()}/{len(use)}（{bss6.bin_hz(use)}）")
    print(f"{'方法':26s} 逆のビン（−40dB以内）  エネルギーの割合  SDR [dB]（音源1, 音源2）  正解で並べ替えた SDR の平均")
    pk = 10 ** (-10 / 20)
    save(OUT / "perm_mix.wav", mix[0] / np.max(np.abs(mix[0])) * pk, fs)
    tags = {"FDICA（そろえない）": "fdica_raw", "FDICA＋パワー比でそろえる": "fdica_pr",
            "FDICA＋正解でそろえる（oracle）": "fdica_oracle", "AuxIVA": "auxiva", "ILRMA（K=2）": "ilrma"}
    for name, (Y, perms) in res.items():
        y = bss6.to_time(Y, n)
        sdr, sir, sar, perm = fast_bss_eval.bss_eval_sources(img, y, compute_permutation=True)
        frac, cnt, wfrac = bss6.swapped_stats(perms, Edb)
        sdr_o = np.mean(bss6.evaluate(img, bss6.to_time(bss6.permute(Y, perms), n))[0])
        print(f"{name:26s} {frac:6.3f}（{cnt:4d} 個）     {wfrac:8.4f}        "
              f"{sdr[0]:6.2f}, {sdr[1]:6.2f}            {sdr_o:6.2f}")
        for k in range(2):                                   # 音源 k に当たる出力を y{k+1} として保存
            yk = y[perm[k]]
            save(OUT / f"perm_{tags[name]}_y{k + 1}.wav", yk / np.max(np.abs(yk)) * pk, fs)

    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(12, 7))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.4])
    ax0 = fig.add_subplot(gs[0, :])
    names = list(res)
    from matplotlib.colors import ListedColormap
    Mp = np.stack([res[nm][1][:, 0] for nm in names]).astype(float)   # 音源 1 に当たる出力の番号（0 か 1）
    Mp[:, ~use] = 2                                                     # エネルギーの小さいビンは灰色
    fbin = np.arange(len(use) + 1) * fs / (2 * (len(use) - 1)) - fs / (4 * (len(use) - 1))
    ax0.pcolormesh(fbin, np.arange(len(names) + 1) - 0.5, Mp, cmap=ListedColormap(["#3b4cc0", "#b40426", "#cccccc"]),
                   vmin=-0.5, vmax=2.5, rasterized=True)
    ax0.set_ylim(len(names) - 0.5, -0.5)
    ax0.set_xlim(0, fs / 2)
    ax0.set_yticks(range(len(names)))
    ax0.set_yticklabels(names)
    ax0.set_xlabel("周波数 [Hz]")
    ax0.set_title("音源 1 が何番目の出力に出たか（青・赤が変わるところで順番が入れ替わる。灰色はエネルギーが −40 dB 未満のビン）")
    ax1 = fig.add_subplot(gs[1, 0])
    specshow(res["FDICA（そろえない）"][0][:, :, 0], fs, bss6.S_HOP, ax=ax1, db_range=60,
             title="FDICA（そろえない）：出力 1 そのまま")
    ax2 = fig.add_subplot(gs[1, 1])
    Yi, pi = res["ILRMA（K=2）"]
    specshow(bss6.permute(Yi, pi)[:, :, 0], fs, bss6.S_HOP, ax=ax2, db_range=60,
             title="ILRMA：音源 1 に当たる出力")
    fig.tight_layout()
    fig.savefig(OUT / "perm_map.png", dpi=150)
    plt.show()   # ノートブック版：図を表示する
    print("\nout/perm_map.png と out/perm_*.wav を保存しました（聞く前に音量を小さく）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"−40 dB 以内のビン (\d+)/") - 935) <= 5, f"「逆のビン」は −40 dB 以内のビン（教科書 935 本前後、0〜7570 Hz）だけで数える（{num(r'−40 dB 以内のビン (\d+)/'):.0f} 本）")
row = lambda name: [float(v) for v in re.search(rf"^{re.escape(name)}\s+([\d.]+)（\s*(\d+) 個）\s+([\d.]+)\s+([-\d.]+),\s+([-\d.]+)\s+([-\d.]+)$", cap.stdout, re.M).groups()]
raw, pr, orc, aux, ilr = (row(n) for n in ("FDICA（そろえない）", "FDICA＋パワー比でそろえる", "FDICA＋正解でそろえる（oracle）", "AuxIVA", "ILRMA（K=2）"))
ok(0.1 <= raw[0] <= 0.4 and abs((raw[3] + raw[4]) / 2) < 2, f"FDICA（そろえない）の逆のビンが 0.1〜0.4（{raw[0]}、{raw[1]:.0f} 本。教科書 0.252、236 本）で、SDR が 0 dB 前後（{raw[3]}, {raw[4]}）")
ok(min(pr[3:5]) > 15 and abs(sum(pr[3:5]) - sum(orc[3:5])) / 2 < 1.5, f"パワー比でそろえると 15 dB 以上（{pr[3]}, {pr[4]}）で、正解でそろえた値（{orc[3]}, {orc[4]}）に近い")
ok(aux[1] == 5 and abs(aux[2] - 0.045) < 0.01, f"AuxIVA は 5 本（{aux[1]:.0f} 本）だが、エネルギーの割合では 4.5%（{aux[2] * 100:.1f}%）")
ok(abs((aux[3] + aux[4]) / 2 - 11.93) < 0.1 and abs(aux[5] - 13.78) < 0.1, f"AuxIVA の SDR は正解で並べ替えると 11.93 → 13.78 dB に上がる（{(aux[3] + aux[4]) / 2:.2f} → {aux[5]}）")
ok(ilr[1] == 1 and ilr[2] < 1e-3 and abs((ilr[3] + ilr[4]) / 2 - ilr[5]) < 0.01 and abs(ilr[5] - 18.57) < 0.1, f"ILRMA は 1 本（エネルギーはほぼ 0）で、並べ替えても 18.57 dB のまま（{ilr[5]}）")
import os
ok(all(os.path.exists(f"out/perm_{t}_y{k}.wav") for t in ("fdica_raw", "fdica_pr", "fdica_oracle", "auxiva", "ilrma") for k in (1, 2)), "out/perm_*.wav ができた")

聞き比べます（手順 E の 4、音量は小さく）：混合音 → FDICA（そろえない）→ パワー比でそろえる → 正解でそろえる → AuxIVA → ILRMA。y1 は男性の声 libri1 に当たる出力です。fdica_raw は、ことばの一部が別の人の声に入れ替わったような、ざらついた音になります。聞こえ方は再生の環境と人によって違うので、ノートに自分の言葉で書いてください。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['out/perm_mix.wav', 'out/perm_fdica_raw_y1.wav', 'out/perm_fdica_pr_y1.wav', 'out/perm_fdica_oracle_y1.wav', 'out/perm_auxiva_y1.wav', 'out/perm_ilrma_y1.wav']:
    print(f)
    listen_file(f)

## 5. 音源分離の論文の作法をまねて評価する（IV-6-3-5・3-9・3-10、手順 F、`eval_iv6.py`）

**ノートブックの既定は試しの設定**（`--seeds 2`：RT60 0.15 s と 0.3 s × 種 2 = 4 試行。執筆時の Mac で 1〜2 分）です。教科書の本番（種 10 通り、20 試行）にするときは、下のセルの `sys.argv` の行をコメントの行に取りかえてください（執筆時の Mac で約 6 分、Colab ではその数倍かかることがあります）。種 2 つでは、教科書の平均 ± 標準偏差・95% 区間・ウィルコクソンの p とは合いません（種 2 つの p は最小でも 0.5 です）。

注意：途中の SDR は `out/eval_trace.csv` に書かれ、次の 6 節（手順 G）がそれを使います。

oracle の切り分け（手順 F の 4）：表の oracleW・oracleR+IP・FDICA+oracleP を、それぞれの推定法（ILRMA・FDICA+PR）と比べ、「失敗の原因が順番そろえか、音源モデルか、周波数ごとの線形分離そのものの限界か」を、残響時間ごとにノートに書いてください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 本番（教科書の手順 F の 3）は次の行の # を外し、その下の sys.argv の行を消す
# import sys; sys.argv = ["eval_iv6.py", "--force"]   # 種 10 通り・20 試行
import sys
sys.argv = ['eval_iv6.py', '--seeds', '2', '--force']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-6：音源分離の論文の作法をまねて、FDICA・AuxIVA・ILRMA と oracle を比べる

使い方: cd ~/home-lab/partIV/iv6 && uv run python eval_iv6.py [--rt60 0.15 0.3] [--seeds 10] [--iter 100]
  作法（音源分離の論文の作法をまねたもの）：pyroomacoustics の鏡像法、16 kHz、残響時間 2 条件、乱数の種 10 通りの平均 ± 標準偏差、
  正解を使う理想条件（oracle）で上限を出し、「音源モデルのせいか、推定のせいか」を切り分ける。
  種 s は、部屋の大きさ・音源とマイクの位置・ILRMA の初期値をまとめて決める（音源は libri1 と libri3 で固定）。
  条件（残響時間・音源）は元の論文と同じではない。論文の T60 = 0 ms と 256 ms にしたいときは --rt60 0 0.256
  （論文の音源は LibriTTS などのコーパス。ここでは librosa の例の音源 2 つで固定）。
方法
  FDICA         : 周波数ごとの ICA（順番をそろえない）
  FDICA+PR      : パワー比の相関で順番をそろえる（ブラインド）
  FDICA+oracleP : 正解で順番をそろえる（パーミュテーションが完全に解けたときの FDICA）
  AuxIVA        : 第7章の auxiva
  AuxIVA+oracleP: AuxIVA の出力を正解で並べ替えたもの（AuxIVA に残ったパーミュテーションの分を見る）
  ILRMA         : 第7章の ilrma（bss6.ilrma_v は同じ計算。途中の反復の SDR も記録する）
  ILRMA+oracleP : ILRMA の出力を正解で並べ替えたもの（AuxIVA+oracleP と比べるときはこちらどうしで）
  ILRMA(LSPB)   : 第7章の ilrma と同じ W を、pyroomacoustics の最小二乗のプロジェクションバックで戻したもの
  ILRMA(pra)    : pyroomacoustics の ilrma（プロジェクションバックも pra の既定）
  oracleR+IP    : 音源モデルに正解のパワーを入れた IP（厳密な上限ではない）
  oracleW       : 正解の像に最小二乗で合わせた分離行列（周波数ごとの線形分離の上限の目安）
出力：out/eval_iv6.csv（1 行＝条件×種×方法、音源 2 つの平均。perm_w は AuxIVA・ILRMA で、順番が多数派と違うビンの
      エネルギーの割合（−40 dB 以内のビン））、out/eval_trace.csv（ILRMA の途中の SDR）、
      out/eval_bars.png、画面に平均 ± 標準偏差と、ILRMA − AuxIVA の対応のある比較
計算時間の目安：執筆時の計算機が空いているとき 1 試行（8 秒の音）15〜19 秒、20 試行で約 6 分
（ほかの処理と重なったときは 1 試行 80〜100 秒、約 32 分）。--out を変えても、途中の SDR は同じフォルダの eval_trace.csv に書く
"""
import argparse
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import pyroomacoustics as pra
from scipy import stats

import bss6
import common.plotting  # noqa: F401
from bss import demix, projection_back, auxiva

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/
OUT.mkdir(exist_ok=True)
ORDER = ["FDICA", "FDICA+PR", "FDICA+oracleP", "AuxIVA", "AuxIVA+oracleP", "ILRMA", "ILRMA+oracleP", "ILRMA(LSPB)", "ILRMA(pra)",
         "oracleR+IP", "oracleW"]


def run_trial(src, fs, rt60, seed, a, trace):
    mix, img = bss6.simulate(src, fs, rt60, seed)
    n = mix.shape[1]
    X = bss6.to_stft(mix)
    S = bss6.to_stft(img)
    F, T, M = X.shape
    sdr_in = float(np.mean(bss6.evaluate(img, np.stack([mix[0], mix[0]]))[0]))
    out = {}

    def timed(name, fn):
        t0 = time.time()
        Y = fn()
        out[name] = (Y, time.time() - t0)

    t0 = time.time()
    Wf = bss6.fdica(X, a.iter)
    out["FDICA"] = (projection_back(demix(Wf, X), Wf), time.time() - t0)
    Yraw = out["FDICA"][0]
    out["FDICA+PR"] = (bss6.permute(Yraw, bss6.align_power_ratio(Yraw)), out["FDICA"][1])
    out["FDICA+oracleP"] = (bss6.permute(Yraw, bss6.align_oracle(Yraw, S)), out["FDICA"][1])
    timed("AuxIVA", lambda: auxiva(X, a.iter)[0])
    Edb = bss6.energy_db(S)
    pw = {}
    pa = bss6.align_oracle(out["AuxIVA"][0], S)
    pw["AuxIVA"] = bss6.swapped_stats(pa, Edb)[2]
    out["AuxIVA+oracleP"] = (bss6.permute(out["AuxIVA"][0], pa), out["AuxIVA"][1])

    ck_sec = [0.0]

    def ck(it, W):                                       # 途中の SDR（この時間は ILRMA の時間から除く）
        t1 = time.time()
        y = bss6.to_time(projection_back(demix(W, X), W), n)
        trace.append(dict(rt60=rt60, seed=seed, iter=it, sdr=float(np.mean(bss6.evaluate(img, y)[0])), sdr_in=sdr_in))
        ck_sec[0] += time.time() - t1
    t0 = time.time()
    Yi, Wi, _ = bss6.ilrma_v(X, a.K, a.iter, seed=seed, checkpoints=set(a.checkpoints), on_checkpoint=ck)
    out["ILRMA"] = (Yi, time.time() - t0 - ck_sec[0])
    pi = bss6.align_oracle(Yi, S)
    pw["ILRMA"] = bss6.swapped_stats(pi, Edb)[2]
    out["ILRMA+oracleP"] = (bss6.permute(Yi, pi), out["ILRMA"][1])
    Yd = demix(Wi, X)                                    # 同じ W を最小二乗のプロジェクションバックで戻す
    z = pra.bss.projection_back(Yd.transpose(1, 0, 2), X[:, :, 0].T)    # (F, N)
    out["ILRMA(LSPB)"] = (Yd * np.conj(z)[:, None, :], out["ILRMA"][1])

    def pra_ilrma():
        np.random.seed(seed)
        return pra.bss.ilrma(X.transpose(1, 0, 2).copy(), n_iter=a.iter, n_components=a.K).transpose(1, 0, 2)
    timed("ILRMA(pra)", pra_ilrma)
    timed("oracleR+IP", lambda: bss6.ip_oracle(X, S, 20)[0])
    timed("oracleW", lambda: demix(bss6.oracle_w(X, S), X))
    rows = []
    for name in ORDER:
        Y, sec = out[name]
        sdr, sir, sar = bss6.evaluate(img, bss6.to_time(Y, n))
        rows.append(dict(rt60=rt60, seed=seed, method=name, sdr_in=sdr_in, sdr=float(np.mean(sdr)),
                         sir=float(np.mean(sir)), sar=float(np.mean(sar)), sdr_imp=float(np.mean(sdr)) - sdr_in, sec=sec,
                         perm_w=pw.get(name, np.nan)))
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.15, 0.3])
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--K", type=int, default=2)
    ap.add_argument("--sec", type=float, default=8.0)
    ap.add_argument("--checkpoints", type=int, nargs="+", default=[5, 10, 20, 30, 50, 70, 100, 150, 200])
    ap.add_argument("--out", default=str(OUT / "eval_iv6.csv"))
    ap.add_argument("--force", action="store_true", help="同じ名前の結果があっても上書きする")
    a = ap.parse_args()
    a.checkpoints = [c for c in a.checkpoints if c <= a.iter]
    out_csv = pathlib.Path(a.out)
    if out_csv.exists() and not a.force:
        sys.exit(f"{out_csv} はすでにあります（上書きするなら --force）")
    fs = 16000
    src = bss6.sources(sec=a.sec)
    rows, trace = [], []
    for rt60 in a.rt60:
        for seed in range(a.seeds):
            t0 = time.time()
            r = run_trial(src, fs, rt60, seed, a, trace)
            rows += r
            msg = "  ".join(f"{x['method']} {x['sdr_imp']:+.1f}" for x in r)
            print(f"RT60 {rt60:.3f} s 種 {seed}（{time.time() - t0:.0f} 秒）SDR 改善量 [dB]：{msg}", flush=True)
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            pd.DataFrame(trace).to_csv(out_csv.with_name("eval_trace.csv"), index=False)
    summarize(pd.DataFrame(rows))


def summarize(df):
    n_seed = df.seed.nunique()
    print(f"\nSDR 改善量 [dB]：平均 ± 標準偏差（種 {n_seed} 通り）と 1 回の計算時間の中央値 [s]")
    tab = df.groupby(["method", "rt60"])["sdr_imp"].agg(["mean", "std"]).unstack("rt60").reindex(ORDER)
    sec = df.groupby("method")["sec"].median().reindex(ORDER)   # スリープなどで止まった回に引っぱられないよう中央値
    for m in ORDER:
        cells = "  ".join(f"{tab.loc[m, ('mean', r)]:6.2f}" + (f" ± {tab.loc[m, ('std', r)]:4.2f}" if n_seed > 1 else "")
                          for r in sorted(df.rt60.unique()))
        print(f"  {m:14s} {cells}   {sec[m]:6.1f} s")
    print("\n順番が多数派と違うビンのエネルギーの割合（−40 dB 以内のビン、種の平均）")
    for m in ("AuxIVA", "ILRMA"):
        g = df[df.method == m].groupby("rt60")["perm_w"].mean()
        print(f"  {m:8s} " + "  ".join(f"RT60 {r:.3f} s：{v * 100:5.1f}%" for r, v in g.items()))
    if n_seed < 2:
        print("\n種が 2 つ未満なので、対応のある比較（区間・検定）は表示しない")
    else:
        for a_, b_, title in (("ILRMA", "AuxIVA", "ILRMA − AuxIVA"),
                              ("ILRMA+oracleP", "AuxIVA+oracleP", "ILRMA+oracleP − AuxIVA+oracleP（どちらも正解で並べ替え）")):
            print(f"\n{title}（同じ種の差。対応のある比較）")
            for r, g in df.groupby("rt60"):
                p = g.pivot(index="seed", columns="method", values="sdr_imp")
                d = (p[a_] - p[b_]).to_numpy()
                ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=d.std(ddof=1) / np.sqrt(len(d)))
                w = stats.wilcoxon(d)
                print(f"  RT60 {r:.3f} s：平均 {d.mean():+.2f} dB、95% t 区間 [{ci[0]:+.2f}, {ci[1]:+.2f}]、"
                      f"正の種 {np.sum(d > 0)}/{len(d)}、ウィルコクソン p = {w.pvalue:.4f}")
        print("\n同じ W でのプロジェクションバックの違い：ILRMA（W⁻¹）− ILRMA(LSPB)（最小二乗）")
        for r, g in df.groupby("rt60"):
            p = g.pivot(index="seed", columns="method", values="sdr")
            d = (p["ILRMA"] - p["ILRMA(LSPB)"]).to_numpy()
            print(f"  RT60 {r:.3f} s：平均 {d.mean():+.2f} ± {d.std(ddof=1):.2f} dB（最小 {d.min():+.2f}、最大 {d.max():+.2f}）")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    rts = sorted(df.rt60.unique())
    wdt = 0.8 / len(rts)
    for j, r in enumerate(rts):
        mu = [tab.loc[m, ("mean", r)] for m in ORDER]
        sd = [tab.loc[m, ("std", r)] for m in ORDER]
        ax.bar(np.arange(len(ORDER)) + (j - (len(rts) - 1) / 2) * wdt, mu, wdt, yerr=sd, capsize=3,
               label=f"RT60 {r * 1000:.0f} ms")
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels(ORDER, rotation=20)
    ax.set_ylabel("SDR 改善量 [dB]（平均 ± 標準偏差）")
    ax.axhline(0, color="k", lw=0.8)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "eval_bars.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("out/eval_bars.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
rows = re.findall(r"^RT60 ([\d.]+) s 種 (\d+)（\d+ 秒）SDR 改善量 \[dB\]：(.*)$", cap.stdout, re.M)
n_seed = len({s for r, s, m in rows})
ok(len(rows) == 2 * n_seed and {r for r, s, m in rows} == {"0.150", "0.300"}, f"{len(rows)} 行（残響時間 2 × 種 {n_seed}）が表示された")
val = lambda m, name: float(re.search(rf"(?:^|  ){re.escape(name)} ([-+\d.]+)", m).group(1))
ok(all(val(m, "ILRMA") > val(m, "FDICA") and val(m, "FDICA+PR") > val(m, "FDICA") for r, s, m in rows),
   "どの行も ILRMA と FDICA+PR の SDR 改善量が FDICA より大きい")
ok(all(val(m, "oracleW") >= max(val(m, k) for k in ("FDICA", "AuxIVA", "ILRMA")) for r, s, m in rows),
   "どの行も oracleW（正解に最小二乗で合わせた W）が推定法より良い")
ok("ILRMA − AuxIVA（同じ種の差" in cap.stdout and "out/eval_bars.png を保存しました" in cap.stdout, "平均 ± 標準偏差の表と ILRMA − AuxIVA の差が表示され、out/eval_bars.png ができた")
import os
ok(os.path.exists("out/eval_trace.csv"), "途中の SDR が out/eval_trace.csv に書かれた（次の 6 節で使う）")

## 6. 統計の誤用を確かめる（IV-6-3-13、手順 G、`misuse_iv6.py`）

5 節の `out/eval_trace.csv` を使います（2〜3 分）。[1] で「試行ごとに最良の回」（正解を見て止めた。誤り）が「最後（100 回）」以上になり、「1つ抜き」（正しい）はそれ以下になることを見ます。[2] は初期値の種を 10 通りに変え、最良の種だけを報告するとどれだけ良く見えるかを示します（[2] は 5 節の種の数に関係なく、教科書と同じ計算です）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['misuse_iv6.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-6：統計の誤用を、自分の分離の結果で確かめる

使い方: cd ~/home-lab/partIV/iv6 && uv run python misuse_iv6.py [--init-seeds 10]
  （先に eval_iv6.py を実行して out/eval_trace.csv を作っておく）
  [1] 反復の回数を「一番よかった回」で止める：試行ごとに最良の回を選ぶと、平均が良く見える。
      正しくは、選ぶのに使わない試行で評価する（種を 1 つずつ抜いて、残りで回数を選び、抜いた種で評価）
  [2] 初期値の乱数の種を選ぶ：同じ混合音で種を 10 通り変え、「一番よい種」の値と平均 ± 標準偏差を比べる
"""
import argparse
import pathlib

import numpy as np
import pandas as pd

import bss6
from bss import demix, projection_back

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--init-seeds", type=int, default=10)
    ap.add_argument("--iter", type=int, default=100)
    a = ap.parse_args()
    tr = pd.read_csv(OUT / "eval_trace.csv")
    tr["imp"] = tr.sdr - tr.sdr_in
    print("[1] ILRMA の反復の回数の選び方（SDR 改善量 [dB]、種の平均）")
    for r, g in tr.groupby("rt60"):
        p = g.pivot(index="seed", columns="iter", values="imp")
        fixed = p[p.columns.max()].mean()
        cheat = p.max(axis=1).mean()                          # 試行ごとに一番よい回（正解を見て止めた）
        common = p.mean().max()                               # 全部の試行で平均が一番よい回（同じ試行で評価）
        loo = []
        for s in p.index:                                     # 種を 1 つ抜いて回数を選び、抜いた種で評価
            best_it = p.drop(index=s).mean().idxmax()
            loo.append(p.loc[s, best_it])
        print(f"  RT60 {r:.3f} s：最後（{p.columns.max()} 回）{fixed:.2f}、試行ごとに最良の回 {cheat:.2f}（誤り）、"
              f"平均が最良の回 {common:.2f}（同じ試行で選んで評価：誤り）、1 つ抜き {np.mean(loo):.2f}（正しい）")
        print(f"     選ばれた回：試行ごと {p.idxmax(axis=1).tolist()}、平均で {p.mean().idxmax()}")

    print(f"\n[2] 初期値の種を {a.init_seeds} 通りに変える（RT60 0.15 s・部屋の種 0 の同じ混合音）")
    src = bss6.sources(sec=8.0)
    mix, img = bss6.simulate(src, 16000, 0.15, 0)
    n = mix.shape[1]
    X = bss6.to_stft(mix)
    sdr_in = float(np.mean(bss6.evaluate(img, np.stack([mix[0], mix[0]]))[0]))
    v = []
    for s in range(a.init_seeds):
        Y, _, _ = bss6.ilrma_v(X, 2, a.iter, seed=s)
        v.append(float(np.mean(bss6.evaluate(img, bss6.to_time(Y, n))[0])) - sdr_in)
        print(f"  種 {s}：{v[-1]:.2f} dB", flush=True)
    v = np.array(v)
    print(f"  平均 ± 標準偏差 {v.mean():.2f} ± {v.std(ddof=1):.2f} dB、最良 {v.max():.2f}、最悪 {v.min():.2f}"
          f"（最良の種だけを報告すると {v.max() - v.mean():+.2f} dB 良く見える）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
for m in re.finditer(r"RT60 ([\d.]+) s：最後（\d+ 回）([-\d.]+)、試行ごとに最良の回 ([-\d.]+)（誤り）、平均が最良の回 ([-\d.]+)（同じ試行で選んで評価：誤り）、1 つ抜き ([-\d.]+)（正しい）", cap.stdout):
    r, last, cheat, common, loo = m.group(1), *map(float, m.groups()[1:])
    ok(cheat >= last and loo <= cheat and common <= cheat,
       f"[1] RT60 {r} s：「試行ごとに最良の回」{cheat} が「最後」{last} 以上で、「1つ抜き」{loo} はそれ以下（教科書は種 10 通りで 16.84 と 16.76、5.85 と 5.66）")
v = [float(x) for x in re.findall(r"^  種 \d+：([-\d.]+) dB", cap.stdout, re.M)]
ok(len(v) == 10 and max(v) - min(v) > 2, f"[2] 初期値の種によって SDR が数 dB 違う（最良 {max(v)}、最悪 {min(v)}）")
d = num(r"最良の種だけを報告すると ([-+\d.]+) dB 良く見える")
ok(d > 0 and abs(d - 0.84) < 0.1, f"[2] 最良の種と平均の差が表示される（{d:+.2f} dB。教科書 +0.84 dB）")

## 7. 優決定・劣決定（IV-6-3-11、手順 H、`over_under.py`）

1〜5 分かかります。pyroomacoustics の OverIVA が低い値になることも含めて、ノートに書いてください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['over_under.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-6：優決定（マイク M ＞ 音源 N）と劣決定（M ＜ N）を試す

使い方: cd ~/home-lab/partIV/iv6 && uv run python over_under.py [--seeds 3]
  優決定（M = 4、N = 2、RT60 0.3 s）：(a) マイクを 2 本だけ使う、(b) 主成分分析（PCA）で 2 次元に減らしてから ILRMA、
                                      (c) pyroomacoustics の OverIVA（pra.bss.auxiva(n_src=2)。ラプラス型とガウス型）
  劣決定（M = 2、N = 3、RT60 0.15 s）：ILRMA（出力は 2 つしかない）、oracleW（3×2 の最小二乗、線形分離の上限）、
                                      理想比マスク（正解を使う非線形の上限）
"""
import argparse
import itertools

import numpy as np
import pyroomacoustics as pra

import bss6
from bss import demix, projection_back, auxiva, ilrma


def sdr_mean(img, Y, n):
    return float(np.mean(bss6.evaluate(img, bss6.to_time(Y, n))[0]))


def pca_ilrma(X, N, K, n_iter, seed):
    """周波数ごとに共分散の大きい N 個の固有ベクトルへ射影 → ILRMA → 全体の分離行列の擬似逆行列で 0 番マイクへ戻す"""
    F, T, M = X.shape
    C = np.einsum("ftm,ftk->fmk", X, X.conj()) / T
    _, E = np.linalg.eigh(C)                                 # 固有値の小さい順
    P = np.conj(np.swapaxes(E[:, :, -N:], 1, 2))             # (F, N, M)：z = P x
    Z = np.einsum("fnm,ftm->ftn", P, X)
    _, Wz = ilrma(Z, K, n_iter, seed=seed)
    Wt = Wz @ P                                              # (F, N, M)：y = Wt x
    A = np.linalg.pinv(Wt)                                   # (F, M, N)：混合行列の推定
    return demix(Wt, X) * A[:, 0, :][:, None, :]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--iter", type=int, default=100)
    a = ap.parse_args()
    fs = 16000
    print("優決定：M = 4（間隔 4 cm）、N = 2、RT60 0.3 s。SDR [dB]（2 音源の平均）")
    src = bss6.sources(("libri1", "libri3"), sec=8.0)
    res = {}
    for seed in range(a.seeds):
        mix, img = bss6.simulate(src, fs, 0.3, seed, n_mics=4)
        n = mix.shape[1]
        X = bss6.to_stft(mix)
        sdr_in = float(np.mean(bss6.evaluate(img, np.stack([mix[0], mix[0]]))[0]))
        r = {"混合音": sdr_in,
             "2 本だけ・AuxIVA": sdr_mean(img, auxiva(X[:, :, :2], a.iter)[0], n),
             "2 本だけ・ILRMA": sdr_mean(img, ilrma(X[:, :, :2], 2, a.iter, seed=seed)[0], n),
             "PCA で 2 次元・ILRMA": sdr_mean(img, pca_ilrma(X, 2, 2, a.iter, seed), n),
             "OverIVA（pra、ラプラス）": sdr_mean(img, pra.bss.auxiva(X.transpose(1, 0, 2).copy(), n_src=2,
                                                                    n_iter=a.iter).transpose(1, 0, 2), n),
             "OverIVA（pra、ガウス）": sdr_mean(img, pra.bss.auxiva(X.transpose(1, 0, 2).copy(), n_src=2, model="gauss",
                                                                  n_iter=a.iter).transpose(1, 0, 2), n)}
        print(f"  種 {seed}：" + "  ".join(f"{k} {v:.2f}" for k, v in r.items()), flush=True)
        for k, v in r.items():
            res.setdefault(k, []).append(v)
    for k, v in res.items():
        print(f"  {k:22s} 平均 {np.mean(v):6.2f}" + (f" ± {np.std(v, ddof=1):4.2f}" if len(v) > 1 else ""))

    print("\n劣決定：M = 2、N = 3（libri1・libri2・libri3）、RT60 0.15 s。SDR [dB]")
    src3 = bss6.sources(("libri1", "libri2", "libri3"), sec=8.0)
    res = {}
    for seed in range(a.seeds):
        mix, img = bss6.simulate(src3, fs, 0.15, seed, n_mics=2)
        n = mix.shape[1]
        X = bss6.to_stft(mix)
        S = bss6.to_stft(img)
        sdr_in = float(np.mean(bss6.evaluate(img, np.stack([mix[0]] * 3))[0]))
        Yi, _ = ilrma(X, 2, a.iter, seed=seed)
        yi = bss6.to_time(Yi, n)
        best = max(np.mean(bss6.evaluate(img[list(p)], yi)[0]) for p in itertools.combinations(range(3), 2))
        Wo = bss6.oracle_w(X, S)                             # (F, 3, 2)
        P = np.abs(S) ** 2
        irm = P / (P.sum(axis=2, keepdims=True) + 1e-12) * X[:, :, :1]
        r = {"混合音（3 音源の平均）": sdr_in,
             "ILRMA（出力 2 つ。よい 2 音源だけ）": float(best),
             "oracleW（3×2 の線形）": sdr_mean(img, demix(Wo, X), n),
             "理想比マスク（非線形）": sdr_mean(img, irm, n)}
        print(f"  種 {seed}：" + "  ".join(f"{k} {v:.2f}" for k, v in r.items()), flush=True)
        for k, v in r.items():
            res.setdefault(k, []).append(v)
    for k, v in res.items():
        print(f"  {k:28s} 平均 {np.mean(v):6.2f}" + (f" ± {np.std(v, ddof=1):4.2f}" if len(v) > 1 else ""))
    print("  （劣決定では、周波数ごとの線形の分離は 1 つの出力で M − 1 = 1 個の音しか消せない。"
          "マスク・多チャネル NMF・深層学習が要る）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
over, under = cap.stdout.split("劣決定：")
avg = lambda txt, k: num(rf"(?m)^  {re.escape(k)}\s+平均\s+([-\d.]+)", text=txt)
i2, ip = avg(over, "2 本だけ・ILRMA"), avg(over, "PCA で 2 次元・ILRMA")
ok(abs(i2 - ip) < 1.0 and abs(i2 - 6.21) < 0.1 and abs(ip - 6.36) < 0.1, f"優決定で、2本だけの ILRMA と PCA で2次元にした ILRMA が同じくらい（{i2}・{ip}。教科書 6.21・6.36 dB）")
ov = avg(over, "OverIVA（pra、ラプラス）")
ok(ov < i2 - 3, f"pyroomacoustics の OverIVA（ラプラス）が低い値（{ov} dB。教科書 0.33 dB）")
u = [avg(under, k) for k in ("ILRMA（出力 2 つ。よい 2 音源だけ）", "oracleW（3×2 の線形）", "理想比マスク（非線形）")]
ok(u[0] < u[1] < u[2] and all(abs(a - b) < 0.1 for a, b in zip(u, (4.66, 8.25, 11.79))), f"劣決定で、ILRMA・oracleW・理想比マスクの順に良くなる {u}（教科書 4.66・8.25・11.79 dB）")

## 8. 確認問題の答え合わせ（7 節、`quiz_check.py`）

確認問題（教科書の 7 節）は、**先に紙で解いてから**実行してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-6：確認問題の答え合わせ（数値の問題だけ）

使い方: cd ~/home-lab/partIV/iv6 && uv run python quiz_check.py
"""
import numpy as np
from scipy import stats

np.set_printoptions(precision=4, suppress=True)

# 問1 混ぜた信号の尖度
a, b = 0.6, 0.8
print("問1 尖度 =", 3 + 3 * a ** 4 + 3 * b ** 4, " 超過尖度 =", 3 * a ** 4 + 3 * b ** 4)

# 問2 白色化
C = np.array([[2.0, 1.0], [1.0, 2.0]])
d, E = np.linalg.eigh(C)
V = np.diag(d ** -0.5) @ E.T
print("問2 固有値", d, " V =\n", V, "\n  V C Vᵀ =\n", V @ C @ V.T)

# 問3 補助関数の数値
r0, r = 2.0, 3.0
print("問3 G(r)=r：G(3) =", r, " 上界 r²/(2r0) + r0/2 =", r ** 2 / (2 * r0) + r0 / 2,
      " r = 2 での上界 =", 2.0 ** 2 / (2 * r0) + r0 / 2)

# 問4 IP の手計算
def ip_row(W, Vn, n):
    w = np.linalg.solve(W @ Vn, np.eye(2)[:, n])
    w = w / np.sqrt(w @ Vn @ w)
    W = W.copy(); W[n] = w
    return W
def Q(W, Vs):
    return sum(W[n] @ Vs[n] @ W[n] for n in range(2)) - 2 * np.log(abs(np.linalg.det(W)))
V1 = np.array([[2.0, 1.0], [1.0, 2.0]]); V2 = np.array([[1.0, 0.0], [0.0, 4.0]])
W = np.eye(2)
print("問4 Q(I) =", Q(W, [V1, V2]))
W = ip_row(W, V1, 0); print("  w1 =", W[0], " Q =", round(Q(W, [V1, V2]), 6))
W = ip_row(W, V2, 1); print("  w2 =", W[1], " Q =", round(Q(W, [V1, V2]), 6), " det W =", round(np.linalg.det(W), 6))
print("  w1ᵀ V2 w2 =", W[0] @ V2 @ W[1])

# 問5 複素数のヤコビアン
Wc = np.array([[1.0, 1j], [0.0, 2.0]])
print("問5 |det W| =", abs(np.linalg.det(Wc)), " 2 log|det W| =", 2 * np.log(abs(np.linalg.det(Wc))))
w = 1.5 - 0.5j
print("  1×1 の例：w = 1.5−0.5j で、面積は |w|² =", abs(w) ** 2, "倍")

# 問6 正規化で J が変わらない
F, T, lam = 1025, 251, 2.0
print("問6 Σlog r の変化 −2FT log λ =", -2 * F * T * np.log(lam), "、−2T Σ_f log|det W| の変化 +2TF log λ =", 2 * T * F * np.log(lam))

# 問7 プロジェクションバック
W = np.array([[1.0, 0.5], [0.25, 1.0]])
A = np.linalg.inv(W)
print("問7 W⁻¹ =\n", A, "\n  y1 を 0 番マイクに戻す係数 (W⁻¹)₀₀ =", A[0, 0], "、y2 は (W⁻¹)₀₁ =", A[0, 1])
W3 = W.copy(); W3[0] *= 3
print("  1 行目を 3 倍したときの係数 =", np.linalg.inv(W3)[0, 0], "（3 倍の y1 に掛けると同じ像）")

# 問8 パーミュテーションの組み合わせの数
print("問8 log10(2^1025) =", 1025 * np.log10(2), "、log10(6^1025) =", 1025 * np.log10(6))

# 問9 ウィルコクソンの最小の p と標準誤差
print("問9 n=10 の両側 p の最小 =", 2 / 2 ** 10, " scipy:", stats.wilcoxon(np.arange(1, 11)).pvalue)
sd = 1.2
print("  SD 1.2 dB・10 種の標準誤差 =", sd / np.sqrt(10), " t の 95% 区間の半幅 =", stats.t.ppf(0.975, 9) * sd / np.sqrt(10))

# 問10 劣決定：1 つの出力で消せる音の数
print("問10 M = 2 の出力 y = wᴴ x：wᴴ a_k = 0 を満たせる k の数は最大 M − 1 = 1")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"問1 尖度 = ([\d.]+)") - 4.6176) < 1e-9, "問1 尖度 4.6176")
ok("[[1. 0.]\n [0. 1.]]" in cap.stdout, "問2 V C Vᵀ = I")
ok(num(r"上界 r²/\(2r0\) \+ r0/2 = ([\d.]+)") == 3.25 and num(r"r = 2 での上界 = ([\d.]+)") == 2.0, "問3 r = 3 で 3.25、r = 2 で 2（等号）")
ok("Q = 5.405465" in cap.stdout and "Q = 3.098612" in cap.stdout and abs(num(r"w1ᵀ V2 w2 = ([-\d.e]+)")) < 1e-12, "問4 q が 6 → 5.4055 → 3.0986、w1ᵀ V2 w2 = 0")
ok(abs(num(r"2 log\|det W\| = ([\d.]+)") - 1.386) < 1e-3, "問5 2 log|det W| = 1.386")
ok(abs(num(r"\+2TF log λ = ([\d.]+)") - 356659) < 1, "問6 打ち消し合う2つの項の大きさ 356,659")
ok(abs(num(r"\(W⁻¹\)₀₀ = ([\d.]+)") - 1.1429) < 1e-4 and abs(num(r"\(W⁻¹\)₀₁ = ([-\d.]+)") + 0.5714) < 1e-4 and abs(num(r"3 倍したときの係数 = ([\d.]+)") - 0.3810) < 1e-4, "問7 係数 1.1429・−0.5714、3 倍で 0.3810")
ok(round(num(r"log10\(2\^1025\) = ([\d.]+)"), 1) == 308.6 and round(num(r"log10\(6\^1025\) = ([\d.]+)"), 1) == 797.6, "問8 10^308.6 と 10^797.6")
ok(abs(num(r"両側 p の最小 = ([\d.]+)") - 0.00195) < 1e-5 and abs(num(r"標準誤差 = ([\d.]+)") - 0.379) < 1e-3 and abs(num(r"半幅 = ([\d.]+)") - 0.858) < 1e-3, "問9 p の最小 0.00195、標準誤差 0.379 dB、半幅 0.858 dB")
ok("最大 M − 1 = 1" in cap.stdout, "問10 最大 1")

## おわりに

- 実数の ICA（中心極限定理・白色化・FastICA）、補助関数と IP の導出、第7章・pyroomacoustics との照合、パーミュテーション問題、oracle を使った評価、統計の誤用、優決定・劣決定を、数値と図と音で確かめました。
- 5 節の評価を教科書の本番（種 10 通り）で行うと、平均 ± 標準偏差と、ILRMA − AuxIVA の 95% 区間・ウィルコクソンの p が教科書と比べられます。
- このノートブックに入れていないもの：論文を読む手順 I（3-9 の対応表を手元に置いて、Kitamura et al.（2016）と Ono（2011）を読む）。